# One-Step HALT - revision experiments (v4): 2 datasets x 2 frozen backbones

| combination | what runs | why |
|---|---|---|
| ECU-MSS + CLIP ViT-B/32 | dev + final | main experiment (re-run from scratch with this code) |
| ECU-MSS + DINOv3 ViT-B/16 | dev + final | backbone check (R6-14): knobs re-tuned on ECU-MSS validation; the method is the one chosen with CLIP |
| DeepSeagrass + CLIP | final only | generalisation (R1C5, R6-4): everything frozen from ECU-MSS + CLIP |
| DeepSeagrass + DINOv3 | final only | everything frozen from ECU-MSS + DINOv3 |

**Run everything with one command:** `python run_all.py --shards 5` (Anaconda Prompt, in the folder with this notebook and `run_all.py`). The combination is chosen with the environment variables `HALT_DATASET` (`ecu` / `deepseagrass`) and `HALT_BACKBONE` (`clip` / `dinov3`); `run_all.py` sets them. DeepSeagrass is read from `C:\DeepSeagrass` (`DS_ROOT`), DINOv3 from Hugging Face (`DINO_NAME`, or a local copy in `DINO_LOCAL`).

All results go to new `v4` folders (`C:\journal paper\clip\{dev,final}\v4` for ECU-MSS + CLIP, `C:\journal paper\clip\runs\<combination>\{dev,final}\v4` for the others, `C:\journal paper\clip\summary_v4` for the summary of all four); v1-v3 results are left untouched. Feature caches are reused.

**New outputs in v4** (for the reviewers): macro-F1 table with its own paired tests (important for the imbalanced DeepSeagrass test set), per-class F1 table, mean confusion matrices, teacher calibration on the pool (accuracy, confidence, ECE), a data/split table, a t-SNE map of where each method's added images lie, a photo-id-free same-photograph check plus an exact-duplicate check for DeepSeagrass, and one summary table/figure across the four combinations.

**v3 notes below still apply.**

---
# (v3)

Everything is computed from **cached frozen CLIP features** (only the 512->256->256 head is trained). v3 reuses the v1 feature cache and writes to `<PROJECT>\<STAGE>\v3`.

**What changed in v3 (2 Nov):** the official validation patches turned out to come from the *same photographs* as the training patches (every validation file is tagged `training`; 47% of validation patches have their nearest training patch in the same photograph, vs 0.1% for test). v3 therefore holds out 10% of the training **photographs** (all their patches) as validation and removes them from shots and pool; the test split is unchanged. Because every v2 choice was made on the leaky split, v3 re-runs the whole dev stage, including the sweeps of the original hybrid (w, then tau, as in v1). The self-training/CEAL threshold and the norm_hybrid gate are fixed at their v2 value (0.6) instead of following the original hybrid's tau. Photo identity is now (site, tag, id).

**What changed in v2 (kept in v3)** (see the cell headers): split-leakage diagnostic + `EXCLUDE_OVERLAP` (P0); CEAL cap bug fixed, `random_correct_oracle`, verification-query counts (P1); new selection family designed **after** the v1 dev results and tuned on validation only: hard confidence gate + farthest-point diversity in frozen CLIP space, distinct source photos, rank-normalised score, teacher-initialised student; diversity diagnostics (P2); FixMatch and Mean Teacher on frozen features (P3); 12 pre-specified primary comparisons with Holm, exploratory tests unadjusted, combined LaTeX tables (P4); a pre-specified decision rule that freezes `NEW_METHOD` before the test set is touched (P5a); every csv row carries a configuration fingerprint, so resumed runs cannot silently mix settings.

**Protocol**
- `pseudo` variants are **leakage-free**: candidates are images the teacher *predicts* as class *c*; hard predicted labels. Ground truth is used only to *log* pseudo-label precision.
- `oracle` variants = **verification-based active learning**: an expert answers "is the predicted class correct?" for each candidate in score order; every answer is counted as a query.
- All knobs (tau, w, tau_gate(K), w_norm(K)) and `NEW_METHOD` are chosen **only on validation** in `STAGE='dev'` and frozen to `frozen_config.json`; `STAGE='final'` runs once on **test** with 20 final seeds.
- Adaptive retraining is no longer part of the proposed method; it stays as an ablation row.

**Easiest: run everything with one command** (Anaconda Prompt, in the folder with this notebook and `run_all.py`): `python run_all.py --shards 5`. It runs the smoke test, the dev stage, checks the split and `frozen_config.json`, then the final stage; shards wait for each other automatically; logs go to `logs\`; start the same command again to resume after any interruption. When using `run_all.py`, do not edit `STAGE`, `SMOKE_TEST` or `SHARD_ID` in the notebook.

**By hand instead:** (0) `HALT_SMOKE=1` once (synthetic data, a few minutes, separate folder). (1) Run the DATA cell and check the photo-level split diagnostic (same photograph must be 0% for val and test). (2) `STAGE='dev'`: run all cells. (3) `STAGE='final'`: run all cells again (sweeps skipped, everything loaded from `frozen_config.json`). (4) Copy `latex_tables.tex`, csv files and figures into the paper.
`STAGE` and `SMOKE_TEST` can be set in the first code cell or with the environment variables `HALT_STAGE` / `HALT_SMOKE`.

**Using the whole workstation:** as in v1, start `N_SHARDS` copies with `SHARD_ID = 0..N_SHARDS-1`; shard 0 builds the cache; cells that need everyone's results stop with `WaitForShards` until all shards are done - then re-run that cell. Keep the data and `C:\journal paper\clip` on the NVMe drive.

In [ ]:
# ===============================================================================
# CONFIG  -  One-Step HALT revision experiments, frozen backbone (CLIP ViT-B/32 or DINOv3 ViT-B/16), ECU-MSS or DeepSeagrass   (v4)
# ===============================================================================
import os, sys, re, json, time, random, math, warnings, hashlib
import numpy as np, pandas as pd
from PIL import Image
from scipy import stats
from sklearn.metrics import precision_recall_fscore_support
from sklearn.linear_model import LogisticRegression
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib; import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

SMOKE_TEST = os.environ.get('HALT_SMOKE', '0') == '1'   # True = tiny synthetic run (no images, no CLIP). Writes to a SEPARATE folder, never to the real results
STAGE      = os.environ.get('HALT_STAGE', 'dev')        # 'dev'   : DEV seeds, evaluated on VALIDATION; all sweeps + the method decision happen here and get FROZEN
                                                        # 'final' : FINAL seeds, evaluated ONCE on TEST with everything loaded from frozen_config.json
K_LIST     = [1, 5, 10, 15]
# ---- v4: dataset x backbone (set by run_all.py through HALT_DATASET / HALT_BACKBONE) -------------------------------------------------------------------
#   ecu + clip          : the main experiment (dev + final), re-run from scratch with the v4 code (same settings as v3; the v3 feature cache is reused).
#   ecu + dinov3        : backbone check (dev + final): knobs re-tuned on the ECU-MSS photo-level validation split; NEW_METHOD = the one chosen with CLIP.
#   deepseagrass + X    : generalisation check (final only): EVERYTHING frozen from ecu + X; no tuning on DeepSeagrass.
DATASET  = os.environ.get('HALT_DATASET', 'ecu')         # 'ecu' | 'deepseagrass'
BACKBONE = os.environ.get('HALT_BACKBONE', 'clip')       # 'clip' | 'dinov3'
assert DATASET in ('ecu', 'deepseagrass') and BACKBONE in ('clip', 'dinov3')
if DATASET == 'deepseagrass': assert STAGE == 'final', 'DeepSeagrass is evaluated with settings frozen on ECU-MSS: final stage only'

# seed plan: DEV/FINAL seeds copied from the DINOv3 notebook (fixed before any result). FINAL_SEEDS_EXTRA = 10 more final seeds drawn with
# np.random.default_rng(20261030) BEFORE any v2 result, so that the final test stage can use 20 seeds (rule 3).
DEV_SEEDS   = [94754, 19933, 55662, 49748, 10238, 72259, 76559, 93769, 17629, 33152]
FINAL_SEEDS = [62568, 68751, 34508, 54993, 82586, 25672, 56061, 7797, 41293, 71812]
FINAL_SEEDS_EXTRA = [76469, 47137, 6670, 61690, 21531, 20464, 26468, 47997, 64983, 97960]
N_FINAL_SEEDS = 20                                       # 10 or 20 (20 recommended: ~2x final-stage time, see the timing table of the dev run)
SEEDS = DEV_SEEDS if STAGE == 'dev' else (FINAL_SEEDS + FINAL_SEEDS_EXTRA)[:N_FINAL_SEEDS]
EVAL_SPLIT = 'val' if STAGE == 'dev' else 'test'

ROOT       = r'C:\Ecu dataset'
TRAIN_JSON = os.path.join(ROOT, 'Training_Patches',   'Train.json')       # pool; its labels are used ONLY to draw shots (and, in 'oracle' variants / logging, to check pseudo-labels)
VAL_JSON   = os.path.join(ROOT, 'Validation_Patches', 'validation.json')
TEST_JSON  = os.path.join(ROOT, 'Test_Patches',       'Test.json')
DS_ROOT    = r'C:\DeepSeagrass'                                            # training/ validation/ test/ <class>/<file>.jpg  (+ *_paths.csv)
PROJECT    = r'C:\journal paper\clip' if not SMOKE_TEST else os.path.join(os.environ.get('HALT_SMOKE_DIR', os.path.abspath('smoke_project')))
RUN_TAG    = 'v4'                                                           # all four combinations are run from scratch into ...\<STAGE>\v4 (v1-v3 results stay untouched as backups); feature caches are reused
def combo_root(ds, bb): return PROJECT if (ds, bb) == ('ecu', 'clip') else os.path.join(PROJECT, 'runs', f'{ds}_{bb}')   # ecu+clip keeps the v3 folders
COMBO      = f'{DATASET}_{BACKBONE}'
CACHE_DIR  = os.path.join(PROJECT, 'cache') if (DATASET, BACKBONE) == ('ecu', 'clip') else os.path.join(PROJECT, f'cache_{COMBO}')   # backbone features (ecu+clip: the v1 cache)
OUT_DIR    = os.path.join(combo_root(DATASET, BACKBONE), STAGE, RUN_TAG)    # csv files, LaTeX tables, figures, logs
FROZEN_JSON = os.path.join(combo_root('ecu', BACKBONE), 'dev', RUN_TAG, 'frozen_config.json')   # written by the ECU-MSS dev stage of this backbone, READ by every final stage
REF_FROZEN_JSON = os.path.join(combo_root('ecu', 'clip'), 'dev', RUN_TAG, 'frozen_config.json')  # ECU-MSS + CLIP: source of NEW_METHOD for the other backbone
os.makedirs(CACHE_DIR, exist_ok=True); os.makedirs(OUT_DIR, exist_ok=True)

CLASSES = ['Amphibolis', 'Background', 'Halophila', 'Posidonia'] if DATASET == 'ecu' else ['Background', 'Ferny', 'Rounded', 'Strappy']; N_CLS = len(CLASSES)
label_to_idx = {c: i for i, c in enumerate(CLASSES)}
CLIP_NAME  = 'openai/clip-vit-base-patch32'     # frozen, 512-d image embedding (as in the paper)
DINO_NAME  = 'facebook/dinov3-vitb16-pretrain-lvd1689m'   # frozen DINOv3 ViT-B/16 (768-d CLS token = pooler_output); Hugging Face transformers >= 4.56
DINO_LOCAL = None                                # optional: a local folder with the downloaded DINOv3 model (config.json + weights), used instead of DINO_NAME
BB_NAME    = CLIP_NAME if BACKBONE == 'clip' else (DINO_LOCAL or DINO_NAME)
BB_DIM     = 512 if BACKBONE == 'clip' else 768
BB_LABEL   = 'CLIP ViT-B/32' if BACKBONE == 'clip' else 'DINOv3 ViT-B/16'
BB_NORM    = ([0.4815, 0.4578, 0.4082], [0.2686, 0.2613, 0.2758]) if BACKBONE == 'clip' else ([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
N_AUG = 48                                      # cached augmented backbone views per labeled image (used as 'augmentation' inside episodes)

# head + episodic training: identical to v1 (512 -> 256 -> 256, AdamW, cosine annealing, 4000 episodes) - fixed a priori
HP = dict(hidden_dim=256, output_dim=256, dropout=0.2, lr=3e-4, weight_decay=1e-3, eta_min=1e-5,
          label_smoothing=0.1, n_episodes=4000, k_support=3, view_queries=4, max_per_class_episode=64)

# ---- machine use (RTX PRO 4000 Blackwell, 24 GB): see the notes in the first markdown cell --------------------------------------------------------------
SHARD_ID, N_SHARDS = int(os.environ.get('HALT_SHARD_ID', 0)), int(os.environ.get('HALT_N_SHARDS', 1))   # run_all.py sets these; by hand: start N_SHARDS copies of this notebook (same code, SHARD_ID = 0 .. N_SHARDS-1); each does its own seeds, results are merged automatically
WAIT_SHARDS = os.environ.get('HALT_WAIT', '0') == '1'   # True (set by run_all.py): a shard WAITS for the others at the end of each grid instead of stopping with WaitForShards
IS_MAIN     = SHARD_ID == 0                               # only shard 0 writes the shared outputs (frozen_config.json, tables, figures)
N_WORKERS  = min(16, max(2, (os.cpu_count() or 8) - 2))
EXTRACT_BS = 128
AMP_EXTRACT = True
if device.type == 'cuda': torch.backends.cuda.matmul.allow_tf32 = True; torch.backends.cudnn.allow_tf32 = True
torch.set_num_threads(2 if N_SHARDS > 1 else 4)

# ---- P0: validation split. The OFFICIAL validation patches are cut from the SAME photographs as the training patches (v2 diagnostic: every validation file carries
# the tag 'training', 9111/9113 share a photo key with train, 47% have their nearest train patch in the same photo; test: 0.1% -> test is photo-separate).
# v3 builds a PHOTO-LEVEL validation split: VAL_PHOTO_FRAC of the training photographs (all their patches, from Train.json AND validation.json) are held out as
# validation and removed from shots and pool; the other validation.json patches are dropped. The official test split is unchanged. Same split in dev AND final.
VAL_MODE, VAL_PHOTO_FRAC, VAL_SPLIT_SEED = 'photo', 0.10, 20261102     # 'photo' (use this) or 'official' (leaky v2 behaviour, only for comparison)

# ---- HALT knobs ---------------------------------------------------------------------------------------------------------------------------------------
# ORIGINAL hybrid (kept as ablation): re-tuned on the photo-level validation split exactly as in v1 (weight sweep at tau=0.9, then tau sweep at w*).
# TAU / W_CONF are the DEFAULTS used while sweeping (as in v1); the dev stage overwrites and freezes them.
TAU, W_CONF = 0.90, 0.60
SWEEP_WEIGHTS = [round(x, 1) for x in np.arange(0, 1.01, 0.1)]; SWEEP_TAUS = [0.5, 0.6, 0.7, 0.8, 0.9]
NORM_GATE   = 0.60                              # candidate gate of norm_hybrid (fixed = its v2 value; decoupled from the original hybrid's tau)
PSEUDO_TAU  = 0.60                              # confidence threshold of the self-training / CEAL baselines (fixed = their v2 value)
ADAPT_MAX_ROUNDS, ADAPT_STEP = 3, 1            # adaptive retraining (ablation only): <=3 rounds, +1 pseudo-labeled sample per class per round
# NEW selection family (designed AFTER seeing the v1 dev results; tuned on dev/validation only, per K):
TAU_GATE_GRID = [0.80, 0.90, 0.95, 0.99]        # hard confidence gate, chosen per K on dev
W_NORM_GRID   = [0.0, 0.25, 0.5, 0.75, 1.0]     # weight of the rank-normalised confidence in 'norm_hybrid', chosen per K on dev
TAU_GATE = {K: 0.90 for K in K_LIST}            # defaults, overwritten by the dev sweep (dev) or by frozen_config.json (final)
W_NORM   = {K: 0.50 for K in K_LIST}
DIV_SPACE = 'clip'                              # diversity of the NEW variants is measured in the frozen CLIP space (L2-normalised, 512-d), not in the few-shot head
RELAX_MULT = 3                                  # if fewer gated candidates than needed: fall back to the top RELAX_MULT*r by confidence (logged as n_relaxed)
NONINF_MARGIN_PP = 0.5                          # decision rule: a candidate may not be worse than the baseline by more than this at any K (dev)
CANDIDATES = ['gate_avg_pseudo', 'gate_fps_pseudo', 'gate_fps_photo_pseudo', 'norm_hybrid_pseudo', 'gate_fps_photo_tinit_pseudo']   # ordered from simplest to most complex
NEW_METHOD = None                               # chosen by the pre-specified rule at the end of the dev stage; loaded from frozen_config.json in the final stage

# ---- semi-supervised baselines on frozen features (P3) --------------------------------------------------------------------------------------------------
U_SUB, U_VIEWS, U_BATCH = 4096, 8, 128          # per seed: 4096 random pool images, 8 cached augmented views each, 128 unlabeled images per episode
LAMBDA_U, EMA_DECAY, SEMI_RAMP = 1.0, 0.99, 0.3   # FixMatch unlabeled weight / Mean Teacher EMA decay / sigmoid ramp-up of the unlabeled loss over the first 30% of episodes (both)
TAU_FM_GRID, LAMBDA_MT_GRID = [0.95, 0.99], [1.0, 10.0]   # ONE knob per baseline, chosen per K on dev - the same tuning budget as the proposed method
SEMI_CFG = {K: dict(tau_fm=0.95, lambda_mt=10.0) for K in K_LIST}   # defaults, overwritten by the dev sweep / frozen_config.json
PSEUDO_CAP = 100                                # self-training / CEAL: at most this many (most confident) PSEUDO-labeled pool images per class

if SMOKE_TEST:
    K_LIST, N_AUG = [1, 5], 4
    DEV_SEEDS, FINAL_SEEDS, FINAL_SEEDS_EXTRA = DEV_SEEDS[:3], FINAL_SEEDS[:3], []
    SEEDS = DEV_SEEDS if STAGE == 'dev' else FINAL_SEEDS
    HP = dict(HP, n_episodes=40); TAU_GATE_GRID, W_NORM_GRID = [0.5, 0.9], [0.0, 1.0]; SWEEP_WEIGHTS, SWEEP_TAUS = [0.0, 0.6, 1.0], [0.5, 0.9]
    TAU_GATE = {K: 0.90 for K in K_LIST}; W_NORM = {K: 0.50 for K in K_LIST}
    U_SUB, U_VIEWS, U_BATCH = 64, 2, 16; SEMI_CFG = {K: dict(tau_fm=0.95, lambda_mt=10.0) for K in K_LIST}

NEW_METHOD_FIXED = None
if BACKBONE != 'clip':                          # the selection method itself is NOT re-chosen per backbone: it is the one chosen on ECU-MSS + CLIP
    if not os.path.exists(REF_FROZEN_JSON): raise FileNotFoundError(f'{REF_FROZEN_JSON} missing: run the ECU-MSS + CLIP dev stage first')
    NEW_METHOD_FIXED = json.load(open(REF_FROZEN_JSON))['NEW_METHOD']; assert NEW_METHOD_FIXED, 'ECU-MSS + CLIP dev stage has no NEW_METHOD'
if STAGE == 'final':
    if not os.path.exists(FROZEN_JSON): raise FileNotFoundError(f'{FROZEN_JSON} missing: run the ECU-MSS dev stage for {BACKBONE} first')
    _fz = json.load(open(FROZEN_JSON))
    TAU, W_CONF, NEW_METHOD = _fz['TAU'], _fz['W_CONF'], _fz['NEW_METHOD']
    TAU_GATE = {int(k): v for k, v in _fz['TAU_GATE'].items()}; W_NORM = {int(k): v for k, v in _fz['W_NORM'].items()}; SEMI_CFG = {int(k): v for k, v in _fz['SEMI_CFG'].items()}
    if DATASET == 'ecu': assert (_fz['VAL_MODE'], _fz['VAL_PHOTO_FRAC'], _fz['VAL_SPLIT_SEED']) == (VAL_MODE, VAL_PHOTO_FRAC, VAL_SPLIT_SEED), 'the validation split must be the same as in the dev stage'
    else: VAL_MODE = 'official'                 # DeepSeagrass: no validation is used (nothing is tuned); shots and pool = all DeepSeagrass training patches
    print(f'FINAL stage ({COMBO}): using the values frozen on the ECU-MSS dev stage ({FROZEN_JSON}) ->', {k: _fz[k] for k in ('TAU', 'W_CONF', 'TAU_GATE', 'W_NORM', 'SEMI_CFG', 'NEW_METHOD') if k in _fz})

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
def tick():
    if device.type == 'cuda': torch.cuda.synchronize()
    return time.perf_counter()
print(f"dataset={DATASET} | backbone={BB_LABEL} ({BB_NAME}) | stage={STAGE} | eval split={EVAL_SPLIT} | {len(SEEDS)} seeds | K={K_LIST} | device={device} | tau={TAU} w={W_CONF} | val_mode={VAL_MODE} | smoke={SMOKE_TEST} | out={OUT_DIR}")


## Data, CLIP feature cache, split-leakage diagnostic and photo-level validation split (P0, R4-7)

In [ ]:
# ===============================================================================
# DATA, SOURCE-PHOTO KEYS, SPLIT-LEAKAGE DIAGNOSTIC (P0, R4-7), FEATURE CACHE
# ===============================================================================
_NAME = re.compile(r'^(?P<cls>[A-Za-z]+)_(?P<site>[A-Za-z0-9]+)_(?P<tag>[A-Za-z]+)_(?P<id>\d+)_')
_DS_TRVA = re.compile(r'^Image(?P<id>\d+)_Row(?P<r>\d+)_Col(?P<c>\d+)$')               # DeepSeagrass training / validation: 32 patches (4 rows x 8 cols) per photograph
_DS_TEST = re.compile(r'^CLASS_(?P<k>\d+)_Image_(?P<id>\d+)_Cell_(?P<cell>\d+)$')      # DeepSeagrass test: patch-level labels; CLASS_k = image-level class of the photograph
def src_key(d):
    """(class, site, tag, photo id) of a patch; the photograph = (site, tag, id)"""
    n = os.path.splitext(str(d.get('image') or os.path.basename(d['path'])))[0]
    if DATASET == 'ecu':
        m = _NAME.match(n); return None if m is None else (m['cls'], m['site'], m['tag'], m['id'])
    m = _DS_TRVA.match(n)
    if m: return (d['label'], 'DS', d['split_tag'], m['id'])
    m = _DS_TEST.match(n)
    return None if m is None else (d['label'], 'DS', f"test{m['k']}", m['id'])
_cache_pref = 'clip' if BACKBONE == 'clip' else BACKBONE
CACHE_COMPLETE = all(os.path.exists(os.path.join(CACHE_DIR, f'{_cache_pref}_{s}_tta.npy')) for s in ('train', 'val', 'test')) and os.path.exists(os.path.join(CACHE_DIR, 'aug_train_fp16.npy.done'))
def _check_files(data, what):
    """all files are checked when features still have to be extracted; with a complete cache only a random sample of 500 (much faster on Windows)"""
    items = data if not CACHE_COMPLETE else random.Random(0).sample(data, min(500, len(data)))
    miss = [it['path'] for it in items if not os.path.exists(it['path'])]
    print(f"{what}: {len(data)} entries, {len(miss)} of {len(items)} checked image files not found" + (f"  e.g. {miss[:2]}" if miss else ''))
    if miss: raise FileNotFoundError(f'{len(miss)} images missing for {what}: check the dataset folder, the class folder names and that the zip files were fully extracted')
def load_split(path):
    """ECU-MSS: read a split json and rebuild every image path from the json's own folder: <split folder>/<class>/<file name>"""
    data = json.load(open(path)); folder = os.path.dirname(path)
    for it in data:
        raw = it['path']; it['raw_path'] = raw; it['path'] = os.path.join(folder, it['label'], os.path.basename(raw.replace('\\', '/')))
    _check_files(data, os.path.basename(path)); return data
def load_ds(split_dir, tag):
    """DeepSeagrass: <DS_ROOT>/<split>_paths.csv (file_path, class_label); paths are rebuilt as <DS_ROOT>/<split>/<class>/<file name>"""
    df = pd.read_csv(os.path.join(DS_ROOT, f'{split_dir}_paths.csv'))
    data = [{'path': os.path.join(DS_ROOT, split_dir, lab, os.path.basename(str(p).replace('\\', '/'))), 'label': lab, 'split_tag': tag}
            for p, lab in zip(df['file_path'], df['class_label'])]
    _check_files(data, f'DeepSeagrass {split_dir}'); return data

if SMOKE_TEST:
    rng_ = np.random.default_rng(0)
    if DATASET == 'ecu':
        def _fake(n_per, tag, id_off):   # file names in the real naming scheme, with deliberately shared (class, site, id) keys across splits
            return [{'path': f'{CLASSES[i % N_CLS]}_S{(i // N_CLS) % 3}_{tag}_{id_off + (i // N_CLS) // 4}_patch_{i % 4}.jpg', 'label': CLASSES[i % N_CLS]} for i in range(n_per * N_CLS)]
        DATA = {'train': _fake(150, 'training', 0), 'val': _fake(40, 'training', 30), 'test': _fake(40, 'test', 0)}
    else:                                 # DeepSeagrass naming; every photograph gives 8 patches of one class (train/val) or mixed patches (test)
        def _fake_trva(n_img, tag): return [{'path': f'Image{j}_Row{1 + q // 4}_Col{q % 4}.jpg', 'label': CLASSES[j % N_CLS], 'split_tag': tag} for j in range(n_img) for q in range(8)]
        DATA = {'train': _fake_trva(80, 'training'), 'val': _fake_trva(16, 'validation'),
                'test': [{'path': f'CLASS_{j % 3}_Image_{j}_Cell_{q}.jpg', 'label': CLASSES[(j + q) % N_CLS], 'split_tag': 'test'} for j in range(20) for q in range(8)]}
elif DATASET == 'ecu':
    DATA = {'train': load_split(TRAIN_JSON), 'val': load_split(VAL_JSON), 'test': load_split(TEST_JSON)}
else:
    DATA = {'train': load_ds('training', 'training'), 'val': load_ds('validation', 'validation'), 'test': load_ds('test', 'test')}
Y = {s: np.array([label_to_idx[d['label']] for d in DATA[s]]) for s in DATA}
for s in DATA: print(f"{s:5s}: {len(DATA[s]):6d} patches | per class {np.bincount(Y[s], minlength=N_CLS).tolist()}  ({', '.join(CLASSES)})")

# ---- photo keys -----------------------------------------------------------------------------------------------------------------------------------------
KEY_FULL = {s: [src_key(d) for d in DATA[s]] for s in DATA}                               # (class, site, tag, id)  -> identity of a source photo inside one split
KEY_XS   = {s: [None if k is None else (k[0], k[1], k[3]) for k in KEY_FULL[s]] for s in DATA}   # (class, site, id)  -> compared ACROSS splits (tag dropped)
for s in DATA:
    nk = sum(k is None for k in KEY_FULL[s]); tags = pd.Series([k[2] for k in KEY_FULL[s] if k is not None]).value_counts().to_dict()
    print(f"{s:5s}: {len(set(k[1:] for k in KEY_FULL[s] if k is not None)):6d} source photos | unparsed names {nk} | file-name tags {tags}")
# photo identity = (site, tag, id): the class is NOT part of it (a multi-species photograph gives patches of several classes); the tag IS part of it
# (ECU-MSS: Train.json and validation.json files are tagged 'training', Test.json files 'test'; ids restart in the test images -> different photographs.
#  DeepSeagrass: the ids restart in every split, so the split name is the tag; whether the splits really use different photographs is checked below in CLIP/DINO space)
PHOTO_KEY = {s: [k[1:] if k is not None else ('__' + s, i) for i, k in enumerate(KEY_FULL[s])] for s in DATA}
_pid = {}
PHOTO_TR = np.array([_pid.setdefault(k, len(_pid)) for k in PHOTO_KEY['train']])   # int photo id per train patch

if DATASET == 'ecu':
    # ---- P0 leakage diagnostic on the OFFICIAL splits (kept for the record / the response letter): shared (class, site, id) keys, tag ignored -------------------
    TRAIN_XS = set(k for k in KEY_XS['train'] if k is not None)
    OV_MASK = {s: np.array([k is not None and k in TRAIN_XS for k in KEY_XS[s]]) for s in ('val', 'test')}
    EVAL_XS = set(k for s in ('val', 'test') for k in KEY_XS[s] if k is not None)
    TRAIN_OV = np.array([k is not None and k in EVAL_XS for k in KEY_XS['train']])            # train patches whose key also occurs in val or test
    for s in ('val', 'test'):
        m = OV_MASK[s]; print(f"{s:5s}: {m.sum():6d} / {len(m)} patches ({100 * m.mean():.1f}%) have a (class, site, photo-id) that also occurs in TRAIN | per class {[int(m[Y[s] == c].sum()) for c in range(N_CLS)]}")
    print(f"train: {TRAIN_OV.sum()} / {len(TRAIN_OV)} patches ({100 * TRAIN_OV.mean():.1f}%) share their key with val or test")
    _x = pd.crosstab(pd.Series([k[2] if k else None for s in DATA for k in KEY_FULL[s]], name='file tag'), pd.Series([s for s in DATA for _ in KEY_FULL[s]], name='split'))
    print('file-name tag x split (if val/test files carry the tag "training", their photos came from the ORIGINAL training images -> patch-level split):\n', _x)
else:
    _pp = pd.Series([PHOTO_KEY['train'][i] for i in range(len(PHOTO_KEY['train']))]).value_counts()
    print(f"DeepSeagrass train: patches per photograph min {_pp.min()} / median {int(_pp.median())} / max {_pp.max()}; "
          f"photographs per class {[len({PHOTO_KEY['train'][i] for i in np.where(Y['train'] == c)[0]}) for c in range(N_CLS)]}")

def wait_for(path, what):
    t0 = time.time()
    while not os.path.exists(path):
        if int((time.time() - t0)) % 60 == 0: print(f'  shard {SHARD_ID}: waiting for shard 0 to finish {what} ...')
        time.sleep(5)

# ---- frozen backbone (CLIP ViT-B/32 or DINOv3 ViT-B/16) + cached features (ECU-MSS + CLIP: the v1 cache is reused) ------------------------------------
HALT_DATA_SRC = r"""
import random, torch
from PIL import Image
import torchvision.transforms as T
_norm = T.Normalize(__MEAN__, __STD__)
TTA_TF = [T.Compose([T.Resize((224, 224)), T.ToTensor(), _norm]),
          T.Compose([T.Resize((224, 224)), T.RandomHorizontalFlip(p=1.0), T.ToTensor(), _norm]),
          T.Compose([T.Resize((256, 256)), T.CenterCrop(224), T.ToTensor(), _norm])]
TRAIN_TF = T.Compose([T.RandomResizedCrop(224, scale=(0.5, 1.0), ratio=(0.9, 1.1)), T.RandomHorizontalFlip(0.5), T.RandomVerticalFlip(0.2),
                      T.RandomRotation(30), T.RandomAffine(0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
                      T.ColorJitter(0.3, 0.3, 0.2, 0.1), T.RandomAutocontrast(0.2), T.ToTensor(), _norm])
class ImgViews(torch.utils.data.Dataset):
    def __init__(self, paths, mode, n_aug=48): self.paths, self.mode, self.n_aug = paths, mode, n_aug
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        im = Image.open(self.paths[i]).convert('RGB')
        if self.mode == 'tta': return torch.stack([t(im) for t in TTA_TF])
        random.seed(1000003 + i); torch.manual_seed(1000003 + i)          # deterministic per image, independent of the number of workers
        return torch.stack([TRAIN_TF(im) for _ in range(self.n_aug)])
""".replace('__MEAN__', str(BB_NORM[0])).replace('__STD__', str(BB_NORM[1]))
AUG_MM = None
if not SMOKE_TEST:
    _modname = 'halt_data' if BACKBONE == 'clip' else f'halt_data_{BACKBONE}'          # one helper module per backbone (normalisation differs)
    _hd = os.path.join(PROJECT, _modname + '.py')
    if IS_MAIN or not os.path.exists(_hd): open(_hd + '.tmp', 'w').write(HALT_DATA_SRC); os.replace(_hd + '.tmp', _hd)
    else: wait_for(_hd, 'the helper module')
    sys.path.insert(0, PROJECT)
    import importlib; halt_data = importlib.reload(importlib.import_module(_modname))
    _BB = [None]
    def get_backbone():
        if _BB[0] is None:
            if BACKBONE == 'clip':
                from transformers import CLIPModel
                _BB[0] = CLIPModel.from_pretrained(BB_NAME).float().eval().to(device)
            else:
                from transformers import AutoModel
                _BB[0] = AutoModel.from_pretrained(BB_NAME).float().eval().to(device)
                assert _BB[0].config.hidden_size == BB_DIM, f'{BB_NAME}: hidden size {_BB[0].config.hidden_size} != {BB_DIM}'
            for p in _BB[0].parameters(): p.requires_grad = False
        return _BB[0]
    @torch.no_grad()
    def encode(x):
        with torch.autocast(device.type, dtype=torch.float16, enabled=(device.type == 'cuda' and AMP_EXTRACT)):
            if BACKBONE == 'clip':
                o = get_backbone().get_image_features(pixel_values=x.to(device, non_blocking=True))
                if not torch.is_tensor(o): o = o.pooler_output
            else:
                o = get_backbone()(pixel_values=x.to(device, non_blocking=True)).pooler_output   # DINOv3: CLS token after the final norm
        assert o.shape[-1] == BB_DIM, o.shape
        return o.float()
    def extract_tta(split):
        f = os.path.join(CACHE_DIR, f'{_cache_pref}_{split}_tta.npy')
        if not os.path.exists(f):
            if N_SHARDS > 1 and SHARD_ID != 0: wait_for(f, f'{split} features'); return np.load(f)
            ds = halt_data.ImgViews([d['path'] for d in DATA[split]], 'tta'); dl = torch.utils.data.DataLoader(ds, batch_size=EXTRACT_BS, num_workers=N_WORKERS, pin_memory=True)
            out = []; t0 = time.time()
            for b, x in enumerate(dl):
                B = x.shape[0]; out.append(encode(x.reshape(-1, 3, 224, 224)).view(B, 3, -1).cpu().numpy())
                if b % 20 == 0: print(f'  {BACKBONE} {split} features: {min((b + 1) * EXTRACT_BS, len(ds))}/{len(ds)}  ({(time.time() - t0) / 60:.1f} min)')
            np.save(f + '.tmp.npy', np.concatenate(out)); os.replace(f + '.tmp.npy', f)
        return np.load(f)
    def precache_aug():
        f = os.path.join(CACHE_DIR, 'aug_train_fp16.npy'); flag = f + '.done'; prog = f + '.progress'; N = len(DATA['train'])
        if os.path.exists(flag): return
        if N_SHARDS > 1 and SHARD_ID != 0: wait_for(flag, 'augmentation features'); return
        start = int(open(prog).read()) if (os.path.exists(f) and os.path.exists(prog)) else 0
        mm = np.lib.format.open_memmap(f, mode='r+' if start else 'w+', dtype=np.float16, shape=(N, N_AUG, BB_DIM))
        ds = torch.utils.data.Subset(halt_data.ImgViews([d['path'] for d in DATA['train']], 'aug', N_AUG), range(start, N))
        dl = torch.utils.data.DataLoader(ds, batch_size=4, num_workers=N_WORKERS, pin_memory=True); i = start; t0 = time.time()
        for b, x in enumerate(dl):
            B = x.shape[0]; mm[i:i + B] = encode(x.reshape(-1, 3, 224, 224)).view(B, N_AUG, -1).half().cpu().numpy(); i += B
            if b % 100 == 0: mm.flush(); open(prog, 'w').write(str(i)); print(f'  {BACKBONE} augmentation cache: {i}/{N}  ({(time.time() - t0) / 60:.1f} min)')
        mm.flush(); open(flag, 'w').write('done'); print('augmentation cache finished')
    TTA = {s: extract_tta(s) for s in DATA}
    for s in DATA: assert TTA[s].shape == (len(DATA[s]), 3, BB_DIM), (s, TTA[s].shape, 'cached features do not match the data: delete the cache folder ' + CACHE_DIR)
    precache_aug(); AUG_MM = np.load(os.path.join(CACHE_DIR, 'aug_train_fp16.npy'), mmap_mode='r')
    assert AUG_MM.shape == (len(DATA['train']), N_AUG, BB_DIM), AUG_MM.shape
    _BB[0] = None
    if device.type == 'cuda': torch.cuda.empty_cache()                                   # the backbone is not needed any more
else:
    D_FEAT = BB_DIM; _ctr = rng_.normal(size=(N_CLS, D_FEAT)) * 0.35
    import zlib
    _src = KEY_XS if DATASET == 'ecu' else PHOTO_KEY                                       # ECU smoke: same-key patches look alike ACROSS splits (a planted leak); DS smoke: no leak
    _phot = {s: np.array([zlib.crc32(str(k).encode()) % 1000 if k else i for i, k in enumerate(_src[s])]) for s in DATA}
    _pv = rng_.normal(size=(1000, D_FEAT)) * 0.25                                        # shared per-key component -> same-key patches look alike (like a real same-photo leak)
    TTA = {s: (_ctr[Y[s]][:, None, :] + _pv[_phot[s] % 1000][:, None, :] + rng_.normal(size=(len(Y[s]), 3, D_FEAT)) * 1.1).astype('float32') for s in DATA}
TTA = {s: torch.tensor(v, dtype=torch.float32, device=device) for s, v in TTA.items()}      # [N, 3 views, BB_DIM]
FEAT_DIM = TTA['train'].shape[-1]
CLIPN = {s: F.normalize(TTA[s].mean(1), dim=1) for s in DATA}                              # frozen BACKBONE embedding (mean of 3 TTA views, L2-normalised) -> diversity + diagnostics (name kept from v3)
print(f'{BB_LABEL} feature dimension:', FEAT_DIM, '| TTA views per image:', TTA['train'].shape[1])

# ---- nearest-neighbour check: does an eval patch's nearest TRAIN patch come from the same photograph?  (chance level = the test split, which is photo-separate) ---
@torch.no_grad()
def nn_to_train(split, train_mask=None, bs=2048):
    T_ = CLIPN['train'] if train_mask is None else CLIPN['train'][torch.as_tensor(np.where(train_mask)[0], device=device)]
    tix = np.arange(len(DATA['train'])) if train_mask is None else np.where(train_mask)[0]; sims, idx = [], []
    for a in range(0, CLIPN[split].shape[0], bs):
        s_, i_ = (CLIPN[split][a:a + bs] @ T_.T).max(1); sims.append(s_.cpu()); idx.append(i_.cpu())
    return torch.cat(sims).numpy(), tix[torch.cat(idx).numpy()]
def leak_report(tag, train_mask=None):
    out = {}
    for s in ('val', 'test'):
        sim, nn_i = nn_to_train(s, train_mask); same = np.array([PHOTO_KEY['train'][nn_i[j]] == PHOTO_KEY[s][j] for j in range(len(sim))])
        out[s] = dict(n=len(sim), nn_sim_median=float(np.median(sim)), frac_nn_same_photo=float(same.mean()))
        print(f"[{tag}] {s:4s}: {len(sim):6d} patches | median cosine similarity to the nearest train patch {out[s]['nn_sim_median']:.3f} | "
              f"nearest train patch from the SAME photograph: {100 * out[s]['frac_nn_same_photo']:.1f}%")
    return out
LEAK = {'official': leak_report('official split')}

# ---- photo-level validation split (v3) ------------------------------------------------------------------------------------------------------------------
VAL_SPLIT_INFO = {'mode': VAL_MODE}
if VAL_MODE == 'photo':
    g_tr, g_va = PHOTO_KEY['train'], PHOTO_KEY['val']; cnt = {}
    for k, y_ in zip(g_tr + g_va, np.concatenate([Y['train'], Y['val']])): cnt.setdefault(k, np.zeros(N_CLS, int))[y_] += 1
    rng_v = np.random.default_rng(VAL_SPLIT_SEED); held = set()
    for c in range(N_CLS):                                                                 # stratified by the majority class of the photograph
        gs = sorted([k for k, v in cnt.items() if v.argmax() == c], key=str)
        if not gs: continue                                                                # no photograph has this majority class
        n_h = max(1, int(round(VAL_PHOTO_FRAC * len(gs))))
        held |= {gs[j] for j in rng_v.choice(len(gs), size=n_h, replace=False)}
    tr_h = np.array([k in held for k in g_tr]); va_h = np.array([k in held for k in g_va]); itr, iva = np.where(tr_h)[0], np.where(va_h)[0]
    VAL_SPLIT_INFO.update(n_photos_total=len(cnt), n_photos_held=len(held), n_val_from_train_json=int(len(itr)), n_val_from_val_json=int(len(iva)),
                          n_val_json_dropped=int((~va_h).sum()), n_train_removed=int(tr_h.sum()))
    DATA['val'] = [DATA['train'][i] for i in itr] + [DATA['val'][j] for j in iva]
    Y['val'] = np.concatenate([Y['train'][itr], Y['val'][iva]]).astype(int)
    TTA['val'] = torch.cat([TTA['train'][torch.as_tensor(itr, device=device)], TTA['val'][torch.as_tensor(iva, device=device)]])
    KEY_FULL['val'] = [KEY_FULL['train'][i] for i in itr] + [KEY_FULL['val'][j] for j in iva]
    PHOTO_KEY['val'] = [PHOTO_KEY['train'][i] for i in itr] + [PHOTO_KEY['val'][j] for j in iva]
    CLIPN['val'] = F.normalize(TTA['val'].mean(1), dim=1)
    TRAIN_ELIGIBLE = ~tr_h
    print(f"photo-level validation: {len(held)} of {len(cnt)} photographs held out ({VAL_PHOTO_FRAC:.0%} per majority class, seed {VAL_SPLIT_SEED}) -> {len(Y['val'])} validation patches "
          f"({len(itr)} from Train.json + {len(iva)} from validation.json) | per class {np.bincount(Y['val'], minlength=N_CLS).tolist()}\n"
          f"  removed from shots/pool: {tr_h.sum()} Train.json patches | dropped: {(~va_h).sum()} validation.json patches of non-held-out photographs")
else:
    TRAIN_ELIGIBLE = np.ones(len(DATA['train']), bool); print('VAL_MODE=official: the leaky official validation split is used (comparison only)')
VAL_SPLIT_INFO['n_train_eligible'] = int(TRAIN_ELIGIBLE.sum())
_ELIG_PHOTOS = {PHOTO_KEY['train'][i] for i in np.where(TRAIN_ELIGIBLE)[0]}
OV_MASK = {s: np.array([k in _ELIG_PHOTOS for k in PHOTO_KEY[s]]) for s in ('val', 'test')}  # eval patch from a photograph that also gives shot/pool patches
for s in ('val', 'test'): print(f"{s:4s}: {OV_MASK[s].sum()} / {len(OV_MASK[s])} patches come from a photograph that also supplies shot/pool patches")
LEAK[VAL_MODE + '_split'] = leak_report(f'{VAL_MODE} split, eligible train only', TRAIN_ELIGIBLE); LEAK['val_split_info'] = VAL_SPLIT_INFO

# ---- v4: same-photograph test that does not need photo ids (needed for DeepSeagrass, whose ids restart in every split) -------------------------------------
# For training patches we know which patches share a photograph. Their nearest neighbour INSIDE the same photograph sets the 'same photo' similarity level.
# If validation/test patches came from training photographs, their nearest training patch would be about as similar as that.  base_rate = how often a training
# patch's nearest patch from ANOTHER photograph is already that similar; a clean evaluation split should show about the same rate.
@torch.no_grad()
def photo_nn_profile(tag, train_mask, n_sample=3000, bs=256):
    tix = np.where(train_mask)[0]; T_ = CLIPN['train'][torch.as_tensor(tix, device=device)]; ph = torch.as_tensor(PHOTO_TR[tix], device=device)
    smp = np.random.default_rng(0).choice(len(tix), size=min(n_sample, len(tix)), replace=False); same, other = [], []
    for a in range(0, len(smp), bs):
        j = torch.as_tensor(smp[a:a + bs], device=device); S = T_[j] @ T_.T; S[torch.arange(len(j), device=device), j] = -2.0
        m = ph[j][:, None] == ph[None, :]; neg = torch.full_like(S, -2.0)
        same.append(torch.where(m, S, neg).max(1).values.cpu()); other.append(torch.where(~m, S, neg).max(1).values.cpu())
    same, other = torch.cat(same).numpy(), torch.cat(other).numpy(); same = same[same > -1.5]
    thr = float(np.median(same)) if len(same) else np.nan
    out = dict(train_same_photo_nn_median=thr, train_other_photo_nn_median=float(np.median(other)), base_rate=float((other >= thr).mean()) if len(same) else np.nan)
    print(f"[{tag}] training patches: nearest patch of the SAME photo median {thr:.3f} | of ANOTHER photo median {out['train_other_photo_nn_median']:.3f} | "
          f"another-photo neighbour at least as close as a same-photo one: {100 * out['base_rate']:.1f}% (base rate)")
    for s in ('val', 'test'):
        sim, _ = nn_to_train(s, train_mask); r = float((sim >= thr).mean()); out[s] = dict(nn_sim_median=float(np.median(sim)), frac_as_close_as_same_photo=r)
        warn = r > 2 * out['base_rate'] + 0.05
        print(f"[{tag}] {s:4s}: nearest training patch median {np.median(sim):.3f} | as close as a same-photo patch: {100 * r:.1f}%" +
              ("   <-- WARNING: much higher than the base rate - this split may contain training photographs" if warn else "   (about the base rate: OK)"))
        out[s]['warning'] = bool(warn)
    return out
LEAK['photo_nn_profile'] = photo_nn_profile(f'{VAL_MODE} split', TRAIN_ELIGIBLE)
if DATASET == 'deepseagrass' and not SMOKE_TEST:                                          # exact duplicate files between the splits (computed once, cached)
    _md5f = os.path.join(CACHE_DIR, 'md5_duplicates.json')
    if not os.path.exists(_md5f):
        if IS_MAIN:
            def _md5(p): return hashlib.md5(open(p, 'rb').read()).hexdigest()
            print('  hashing all DeepSeagrass files to look for exact duplicates between the splits (once) ...'); H = {s: [_md5(d['path']) for d in DATA[s]] for s in DATA}
            htr = set(H['train']); dup = {s: int(sum(h in htr for h in H[s])) for s in ('val', 'test')}; dup['train_internal'] = len(H['train']) - len(htr)
            json.dump(dup, open(_md5f + '.tmp', 'w')); os.replace(_md5f + '.tmp', _md5f)
        else: wait_for(_md5f, 'the duplicate check')
    LEAK['exact_duplicates'] = json.load(open(_md5f)); print('exact duplicate files (identical bytes) also found in the training split:', LEAK['exact_duplicates'])
if IS_MAIN: json.dump(LEAK, open(os.path.join(OUT_DIR, 'leakage_diagnostic.json'), 'w'), indent=2)
if VAL_MODE == 'photo':
    _bad = {s: LEAK['photo_split'][s]['frac_nn_same_photo'] for s in ('val', 'test') if LEAK['photo_split'][s]['frac_nn_same_photo'] > 0.005}
    assert not _bad, f'photo-level split check FAILED: nearest train patch from the same photograph {_bad} - stop and report this'
print("READ: with the photo-level split, 'same photograph' must be 0% for val and test, and the val similarity should be close to the test similarity.")
CLIPN_TR_NP = CLIPN['train'].cpu().numpy()

_FULL = {}
def get_full(i):
    """[3 + N_AUG, 512] features of train image i: the 3 TTA views followed by N_AUG random augmentations (read from the cache)."""
    if i in _FULL: return _FULL[i]
    if SMOKE_TEST:
        g = torch.Generator().manual_seed(int(i)); aug = TTA['train'][i, 0][None] + 0.3 * torch.randn(N_AUG, FEAT_DIM, generator=g).to(device)
    else:
        aug = torch.from_numpy(np.asarray(AUG_MM[i], dtype=np.float32)).to(device)
    _FULL[i] = torch.cat([TTA['train'][i], aug], 0); return _FULL[i]
def make_SFc(idx_per_class):
    return [torch.stack([get_full(int(i)) for i in lst]) for lst in idx_per_class]
def aug_views(idx, n_views):
    """[len(idx), n_views, 512] cached augmented views of train images (for FixMatch / Mean Teacher strong views)"""
    idx = np.asarray(idx)
    if SMOKE_TEST:
        g = torch.Generator().manual_seed(int(idx.sum()) % (2 ** 31)); return TTA['train'][idx, :1].repeat(1, n_views, 1) + 0.3 * torch.randn(len(idx), n_views, FEAT_DIM, generator=g).to(device)
    o = np.argsort(idx); arr = np.empty((len(idx), n_views, FEAT_DIM), np.float32); arr[o] = np.asarray(AUG_MM[idx[o], :n_views], dtype=np.float32)
    return torch.from_numpy(arr).to(device)


## Core: shots, episodic prototypical head, metrics (with overlap-subset accuracy)

In [ ]:
# ===============================================================================
# CORE: shots, episodic prototypical head, prediction, metrics
# ===============================================================================
def src_of(i): return KEY_FULL['train'][i]

def sample_shots(K, seed, distinct=True):
    """K random labeled train images per class from the ELIGIBLE train patches; different source photos when possible; nested across K for a seed."""
    rng = random.Random(seed); out = []
    for c in range(N_CLS):
        items = [i for i in range(len(DATA['train'])) if Y['train'][i] == c and TRAIN_ELIGIBLE[i]]; rng.shuffle(items); chosen, seen = [], set()
        for i in items:
            key = src_of(i)
            if distinct and key is not None and key in seen: continue
            chosen.append(i); seen.add(key)
            if len(chosen) == K: break
        if len(chosen) < K: chosen += [i for i in items if i not in set(chosen)][:K - len(chosen)]
        out.append(chosen)
    return out

def build_head():
    return nn.Sequential(nn.Linear(FEAT_DIM, HP['hidden_dim']), nn.ReLU(inplace=True), nn.Dropout(HP['dropout']), nn.Linear(HP['hidden_dim'], HP['output_dim'])).to(device)
def _init_head(init):
    head = build_head()
    if init is not None: head.load_state_dict(init.state_dict())         # teacher-initialised student (P2e)
    return head

def episode_plan(Kc):
    if Kc >= 5: return HP['k_support'], Kc - HP['k_support'], 0
    K_S = max(1, Kc // 2); return K_S, Kc - K_S, HP['view_queries']

def _make_opt(head):
    n_ep = HP['n_episodes']; opt = torch.optim.AdamW(head.parameters(), lr=HP['lr'], weight_decay=HP['weight_decay'])
    return opt, torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_ep, eta_min=HP['eta_min']), nn.CrossEntropyLoss(label_smoothing=HP['label_smoothing'])

def _train_loop(SFc, seed, init=None):
    """Episodic prototypical training on cached views; classes may have different sizes. Fixed #episodes, last checkpoint."""
    set_seed(seed); n_ep = HP['n_episodes']; head = _init_head(init); opt, sched, loss_fn = _make_opt(head)
    V = SFc[0].shape[1]; plans = []
    for c in range(N_CLS):
        Kc = min(SFc[c].shape[0], HP['max_per_class_episode']); K_S, Q_S, QV = episode_plan(Kc); plans.append((Kc, K_S, Q_S, QV))
    for ep in range(n_ep):
        head.train(); xs, sizes = [], []
        for c in range(N_CLS):
            Kc, K_S, Q_S, QV = plans[c]; perm = torch.randperm(SFc[c].shape[0], device=device)[:Kc]; vw = torch.randint(0, V, (Kc,), device=device)
            x = SFc[c][perm, vw]
            if QV > 0:
                vq = torch.randint(0, V, (K_S, QV), device=device); x = torch.cat([x, SFc[c][perm[:K_S, None], vq].reshape(-1, x.shape[-1])], 0)
            xs.append(x); sizes.append(x.shape[0])
        emb = head(torch.cat(xs, 0)).split(sizes); protos, qs, lab = [], [], []
        for c in range(N_CLS):
            Kc, K_S, Q_S, QV = plans[c]; protos.append(emb[c][:K_S].mean(0)); q = emb[c][K_S:]; qs.append(q); lab.append(torch.full((q.shape[0],), c, device=device))
        logits = -torch.cdist(torch.cat(qs, 0), torch.stack(protos)) ** 2; loss = loss_fn(logits, torch.cat(lab))
        opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    head.eval(); return head

def _episode_fast(head, SF, Kc, K_S, QV, ar, lab, loss_fn):
    """one labeled episode for a balanced set SF [C, K, V, D]; returns (loss, prototypes, the sampled episode for re-use)"""
    C, K, V, D = SF.shape; perm = torch.rand(C, K, device=device).argsort(1)[:, :Kc]
    x = SF[ar, perm, torch.randint(0, V, (C, Kc), device=device)]
    e = head(x.reshape(-1, D)).view(C, Kc, -1); protos = e[:, :K_S].mean(1); q = e[:, K_S:]
    if QV > 0:
        xq = SF[ar[:, :, None], perm[:, :K_S, None], torch.randint(0, V, (C, K_S, QV), device=device)]
        q = torch.cat([q, head(xq.reshape(-1, D)).view(C, K_S * QV, -1)], 1)
    return loss_fn(-torch.cdist(q.reshape(-1, q.shape[-1]), protos) ** 2, lab), protos, (x, xq if QV > 0 else None)

def _train_fast(SF, seed, init=None):
    """same episodic training for BALANCED sets, vectorised over classes"""
    set_seed(seed); n_ep = HP['n_episodes']; head = _init_head(init); opt, sched, loss_fn = _make_opt(head)
    C, K, V, D = SF.shape; Kc = min(K, HP['max_per_class_episode']); K_S, Q_S, QV = episode_plan(Kc); ar = torch.arange(C, device=device)[:, None]
    lab = torch.arange(C, device=device).repeat_interleave(Q_S + K_S * QV)
    for ep in range(n_ep):
        head.train(); loss, _, _ = _episode_fast(head, SF, Kc, K_S, QV, ar, lab, loss_fn)
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); sched.step()
    head.eval(); return head

def train_head(SFc, seed, init=None):
    if len({s.shape[0] for s in SFc}) == 1: return _train_fast(torch.stack(SFc), seed, init)
    return _train_loop(SFc, seed, init)

@torch.no_grad()
def class_protos(head, SFc):
    """test-time prototype = mean head embedding of the plain view (view 0) of ALL labeled samples of the class"""
    return torch.stack([head(SFc[c][:, 0]).mean(0) for c in range(N_CLS)])
@torch.no_grad()
def probs(head, X, protos, bs=16384):
    """class probabilities: softmax of -squared distance to the prototypes; embedding = mean over the 3 TTA views.  X: [N, 3, D]"""
    out = []
    for a in range(0, X.shape[0], bs):
        x = X[a:a + bs]; N, Vv, D = x.shape; e = head(x.reshape(-1, D)).view(N, Vv, -1).mean(1); out.append(torch.softmax(-torch.cdist(e, protos) ** 2, 1))
    return torch.cat(out)
@torch.no_grad()
def plain_emb_norm(head, X):
    return F.normalize(head(X[:, 0]), dim=1)

def metrics_from(y, pred, ov=None):
    p, r, f, _ = precision_recall_fscore_support(y, pred, labels=list(range(N_CLS)), zero_division=0)
    m = {'acc': float((pred == y).mean()), 'macro_f1': float(f.mean())}
    for c, n in enumerate(CLASSES): m[f'f1_{n}'], m[f'prec_{n}'], m[f'rec_{n}'] = float(f[c]), float(p[c]), float(r[c])
    m['cm'] = json.dumps(np.bincount(N_CLS * np.asarray(y) + np.asarray(pred), minlength=N_CLS * N_CLS).reshape(N_CLS, N_CLS).tolist())   # rows = true, cols = predicted
    if ov is not None:                                                    # P0: accuracy on eval patches whose photo key also occurs in train vs the rest
        m['acc_ov'] = float((pred[ov] == y[ov]).mean()) if ov.any() else np.nan; m['acc_nov'] = float((pred[~ov] == y[~ov]).mean()) if (~ov).any() else np.nan
    return m
def calibration(P, y, n_bins=15):
    """teacher on the unlabeled pool (ground truth used only for this diagnostic): accuracy, mean confidence, expected calibration error (15 equal-width bins)"""
    conf, pred = P.max(1), P.argmax(1); ok = (pred == y).astype(float); bins = np.minimum((conf * n_bins).astype(int), n_bins - 1)
    ece = sum(abs(ok[bins == b].mean() - conf[bins == b].mean()) * (bins == b).mean() for b in range(n_bins) if (bins == b).any())
    return dict(teacher_pool_acc=float(ok.mean()), teacher_pool_conf=float(conf.mean()), teacher_pool_ece=float(ece),
                teacher_acc_conf_ge_0_9=float(ok[conf >= 0.9].mean()) if (conf >= 0.9).any() else np.nan, teacher_frac_conf_ge_0_9=float((conf >= 0.9).mean()))
def eval_head(head, protos):
    return metrics_from(Y[EVAL_SPLIT], probs(head, TTA[EVAL_SPLIT], protos).argmax(1).cpu().numpy(), OV_MASK[EVAL_SPLIT])

# ---- per (K, seed) context: shots, baseline head (= single-pass few-shot baseline), pool predictions ------------------------------------
CTX = {}; CTX_MAX = 4                                     # small LRU: run_grid loops K -> seed -> variant, so a context is only needed inside one (K, seed)
def get_ctx(K, seed):
    if (K, seed) in CTX: return CTX[(K, seed)]
    while len(CTX) >= CTX_MAX: CTX.pop(next(iter(CTX)))
    shots = sample_shots(K, seed); sset = {i for l in shots for i in l}
    pool = np.array([i for i in range(len(DATA['train'])) if i not in sset and TRAIN_ELIGIBLE[i]])
    SFc = make_SFc(shots); t0 = tick(); head = train_head(SFc, seed); t_train = tick() - t0          # head training only (CLIP features are a one-off cache)
    protos = class_protos(head, SFc); Xp = TTA['train'][pool]
    t0 = tick(); P0 = probs(head, Xp, protos).cpu().numpy(); E0 = plain_emb_norm(head, Xp).cpu().numpy(); t_pool = tick() - t0   # teacher inference on the whole pool
    ctx = dict(K=K, seed=seed, shots=shots, pool=pool, y_pool=Y['train'][pool], photo_pool=PHOTO_TR[pool], SFc=SFc, head0=head, protos0=protos, P0=P0, E0=E0,
               Es0=[F.normalize(head(SFc[c][:, 0]), dim=1).detach().cpu().numpy() for c in range(N_CLS)], t_train=t_train, t_pool=t_pool, base_metrics=eval_head(head, protos))
    ctx['teacher_calib'] = calibration(P0, Y['train'][pool])
    CTX[(K, seed)] = ctx; return ctx


## HALT selection: original family (ablations), new gated diversity family, verification-based oracle variants, diversity diagnostics

In [ ]:
# ===============================================================================
# HALT selection  (pseudo = leakage-free: predicted labels + confidence only  |  oracle = verification-based active learning: an expert answers
# "is the predicted class correct?" for each candidate in score order; only accepted candidates are kept; every answer counts as one QUERY)
# ===============================================================================
# sel:   'random'      random order over the candidates (pseudo: predicted class c; oracle: verified)
#        'random_true' class-balanced random LABELED images (v1 'random_oracle': the expert labels random images of class c; queries = r)
#        'hybrid'      ORIGINAL score  w*Conf + (1-w)*AvgDissim(x, refs)  in head space, refs = shots (+ chosen), recomputed after every pick
#        'avg'         gate, then max AvgDissim to refs (pure diversity among gated candidates)                   [NEW]
#        'fps'         gate, then farthest-point: max over x of min_r (1 - cos(x, r)), refs = shots + chosen      [NEW]
#        'norm'        rank-normalised  w(K)*rank(Conf) + (1-w(K))*rank(AvgDissim)  among the candidates           [NEW]
# gate:  None or a confidence threshold;  photo: True = no two selected images (and no selected image and a shot) from the same source photo
# space: 'head' (teacher head embedding, as in the paper) or 'clip' (frozen CLIP, used by the NEW variants)
# tinit: True = student initialised from the teacher head instead of from scratch
def _ranks(v): o = np.argsort(np.argsort(v)); return o / max(1, len(v) - 1)

def pick_class(c, r, ctx, P, E, refs_head, taken, cfg, rng, info):
    """choose r pool rows for class c; returns a list of pool rows. Updates info (queries, relaxations)."""
    if r <= 0: return []
    pred, conf = P.argmax(1), P.max(1); y = ctx['y_pool']; oracle = cfg['label'] == 'oracle'; sel = cfg['sel']
    if sel == 'random_true':
        cand = np.where(~taken & (y == c))[0]; ch = list(rng.choice(cand, size=min(r, len(cand)), replace=False)) if len(cand) else []
        info['n_queries'] += len(ch); return ch
    base = ~taken & (pred == c)
    gate = cfg.get('gate')
    if gate is None and sel in ('hybrid', 'norm') and cfg.get('tau') is not None: gate = cfg['tau']      # original methods: tau is the candidate gate
    cand = np.where(base & (conf >= gate))[0] if gate is not None else np.where(base)[0]
    if gate is not None and len(cand) < r:                                    # not enough gated candidates -> top RELAX_MULT*r by confidence (logged)
        info['n_relaxed'] += 1; b = np.where(base)[0]
        cand = b if sel == 'hybrid' else b[np.argsort(-conf[b])][:max(RELAX_MULT * r, len(cand))]   # original hybrid: v1 rule (all predicted-c images)
    if cfg.get('photo'):                                                       # P2b: distinct source photos from the shots
        used = set(PHOTO_TR[ctx['shots'][c]].tolist()) | set(PHOTO_TR[np.concatenate(ctx['shots'])].tolist())
        cand = cand[~np.isin(ctx['photo_pool'][cand], list(used))]
    else: used = set()
    if cfg.get('space', 'head') == 'clip':
        X = CLIPN_TR_NP[ctx['pool'][cand]]; R = CLIPN_TR_NP[ctx['shots'][c]]
    else:
        X = E[cand]; R = refs_head
    m = R.mean(0); n = len(R)                                                  # AvgDissim(x, refs) = 1 - x . mean(refs)  (exact for unit vectors)
    mind = 1.0 - (X @ R.T).max(1) if sel == 'fps' else None                    # farthest-point: min cosine distance to refs
    alive = np.ones(len(cand), bool); chosen = []
    order = rng.permutation(len(cand)) if sel == 'random' else None; ptr = 0
    while len(chosen) < r and alive.any():
        if sel == 'random':
            while ptr < len(order) and not alive[order[ptr]]: ptr += 1
            if ptr >= len(order): break
            j = order[ptr]
        else:
            dis = 1.0 - X @ m
            if sel == 'hybrid': sc = cfg['w'] * conf[cand] + (1 - cfg['w']) * dis
            elif sel == 'avg':  sc = dis
            elif sel == 'fps':  sc = mind.copy()
            elif sel == 'norm': sc = cfg['w'] * _ranks(conf[cand]) + (1 - cfg['w']) * _ranks(dis)
            sc = np.where(alive, sc, -np.inf); j = int(np.argmax(sc))
        alive[j] = False; i = cand[j]
        if oracle:                                                             # verification query: is the predicted class c correct?
            info['n_queries'] += 1
            if y[i] != c: continue
        if cfg.get('photo'):
            ph = ctx['photo_pool'][i]
            if ph in used: continue
            used.add(ph); alive &= ctx['photo_pool'][cand] != ph
        chosen.append(int(i)); m = (m * n + X[j]) / (n + 1); n += 1
        if mind is not None: mind = np.minimum(mind, 1.0 - X @ X[j])
    if len(chosen) < r and not oracle:                                         # still short -> most probable remaining images for class c (forced pseudo-label c)
        rest = np.where(~taken)[0]; rest = [int(i) for i in rest[np.argsort(-P[rest, c])] if int(i) not in set(chosen)][:r - len(chosen)]; chosen += rest; info['n_forced'] += len(rest)
    return chosen

def retrain(ctx, extra_rows, seed, init=None):
    """train a head on the shots + extra pool rows (list per class). Returns head, SFc, protos"""
    idx = [list(ctx['shots'][c]) + [int(ctx['pool'][i]) for i in extra_rows[c]] for c in range(N_CLS)]
    global LAST_TRAIN_T
    SFc = make_SFc(idx); t0 = tick(); head = train_head(SFc, seed, init); LAST_TRAIN_T = tick() - t0; return head, SFc, class_protos(head, SFc)

def pool_state(ctx, head, protos):
    Xp = TTA['train'][ctx['pool']]; return probs(head, Xp, protos).cpu().numpy(), plain_emb_norm(head, Xp).cpu().numpy()

def halt_expand(ctx, cfg, seed, k=None):
    """returns chosen pool rows per class (+ info). Optional adaptive-retraining stage first (ablation only; adds only pseudo-labeled samples)."""
    K = ctx['K']; k = K if k is None else k; rng = np.random.default_rng(seed); y_pool = ctx['y_pool']; oracle = cfg['label'] == 'oracle'
    head, protos, P, E = ctx['head0'], ctx['protos0'], ctx['P0'], ctx['E0']
    taken = np.zeros(len(y_pool), bool); chosen = [[] for _ in range(N_CLS)]
    info = dict(adapt_rounds=0, adapt_stop='off', t_adapt=0.0, t_select=0.0, n_relaxed=0, n_forced=0, n_queries=0, mean_conf_start=float(P.max(1).mean()))
    Es = ctx['Es0']
    if cfg.get('adaptive'):
        SFc_cur = ctx['SFc']
        while P.max(1).mean() < cfg['tau'] and info['adapt_rounds'] < ADAPT_MAX_ROUNDS and len(chosen[0]) < k:
            pred, conf = P.argmax(1), P.max(1)
            for c in range(N_CLS):
                msk = ~taken & (pred == c); cand = np.where(msk)[0]; cand = cand[np.argsort(-conf[cand])]; acc = 0
                for i in cand:
                    if acc >= ADAPT_STEP: break
                    if oracle:
                        info['n_queries'] += 1
                        if y_pool[i] != c: taken[i] = True; continue
                    chosen[c].append(int(i)); taken[i] = True; acc += 1
            head, SFc_cur, protos = retrain(ctx, chosen, seed + 17 * (info['adapt_rounds'] + 1)); P, E = pool_state(ctx, head, protos)
            Es = [F.normalize(head(SFc_cur[c][:, 0]), dim=1).detach().cpu().numpy() for c in range(N_CLS)]
            info['adapt_rounds'] += 1; info['t_adapt'] += LAST_TRAIN_T
        info['adapt_stop'] = 'not_needed' if info['adapt_rounds'] == 0 else ('tau_met' if P.max(1).mean() >= cfg['tau'] else 'max_rounds')
    t0 = tick()
    for c in range(N_CLS):
        r = k - len(chosen[c]); new = pick_class(c, r, ctx, P, E, Es[c], taken, cfg, rng, info)
        for i in new: chosen[c].append(int(i)); taken[i] = True
    info['t_select'] = tick() - t0; return chosen, info

def diversity_stats(ctx, chosen):
    """P2 diagnostic in frozen CLIP space: mean pairwise cosine distance of the added images (per class), mean distance to the nearest shot, same-photo rate"""
    pw, dshot, same, tot = [], [], 0, 0
    for c in range(N_CLS):
        if not chosen[c]: continue
        A = CLIPN_TR_NP[ctx['pool'][chosen[c]]]; S = CLIPN_TR_NP[ctx['shots'][c]]
        if len(A) > 1: G = 1 - A @ A.T; pw.append(G[np.triu_indices(len(A), 1)].mean())
        dshot.append((1 - A @ S.T).min(1).mean())
        ph = ctx['photo_pool'][chosen[c]]; seen = set(PHOTO_TR[ctx['shots'][c]].tolist())
        for p in ph: same += p in seen; seen.add(p); tot += 1
    return dict(div_pairwise=float(np.mean(pw)) if pw else np.nan, div_to_shots=float(np.mean(dshot)) if dshot else np.nan, same_photo_rate=same / max(1, tot))

def run_variant(K, seed, name, cfg):
    ctx = get_ctx(K, seed)
    if name == 'baseline':
        row = dict(ctx['base_metrics']); row.update(t_train=ctx['t_train'], t_pool=ctx['t_pool'], t_select=0.0, t_adapt=0.0, t_final=0.0, n_added=0, n_queries=0, **ctx['teacher_calib']); return row
    chosen, info = halt_expand(ctx, cfg, seed)
    head, SFc, protos = retrain(ctx, chosen, seed, ctx['head0'] if cfg.get('tinit') else None); t_final = LAST_TRAIN_T
    row = eval_head(head, protos); rows = [i for c in range(N_CLS) for i in chosen[c]]
    prec_c = [float(np.mean(ctx['y_pool'][chosen[c]] == c)) if len(chosen[c]) else np.nan for c in range(N_CLS)]
    row.update(info); row.update(diversity_stats(ctx, chosen))
    row.update(t_train=ctx['t_train'], t_pool=ctx['t_pool'], t_final=t_final, n_added=len(rows),
               pseudo_precision=float(np.mean([ctx['y_pool'][i] == c for c in range(N_CLS) for i in chosen[c]])) if rows else np.nan,
               **{f'pseudo_prec_{n}': prec_c[c] for c, n in enumerate(CLASSES)}, **{f'n_added_{n}': len(chosen[c]) for c, n in enumerate(CLASSES)}); return row

def V(sel, w=None, label='pseudo', **kw): return dict(sel=sel, w=w, label=label, tau=TAU, **kw)
def variants(K):
    g, wn = TAU_GATE[K], W_NORM[K]
    return {'baseline': None,
            # leakage-free ablations (original family)
            'random_pseudo': V('random'), 'conf_pseudo': V('hybrid', 1.0), 'dissim_pseudo': V('hybrid', 0.0),
            'hybrid_pseudo': V('hybrid', W_CONF), 'hybrid_adapt_pseudo': V('hybrid', W_CONF, adaptive=True),
            # NEW leakage-free family (designed after the v1 dev results, tuned on dev only)
            'random_gate_pseudo': V('random', gate=g),                                     # control: is the gate alone enough?
            'gate_avg_pseudo': V('avg', gate=g, space=DIV_SPACE), 'gate_fps_pseudo': V('fps', gate=g, space=DIV_SPACE),
            'gate_fps_photo_pseudo': V('fps', gate=g, space=DIV_SPACE, photo=True), 'norm_hybrid_pseudo': V('norm', wn, gate=NORM_GATE, space=DIV_SPACE),
            'gate_fps_photo_tinit_pseudo': V('fps', gate=g, space=DIV_SPACE, photo=True, tinit=True),
            # verification-based active learning (oracle answers yes/no for the predicted class; queries are counted)
            'random_oracle': V('random_true', label='oracle'), 'random_correct_oracle': V('random', label='oracle'),
            'conf_oracle': V('hybrid', 1.0, 'oracle'), 'hybrid_oracle': V('hybrid', W_CONF, 'oracle'), 'hybrid_adapt_oracle': V('hybrid', W_CONF, 'oracle', adaptive=True),
            'gate_fps_photo_oracle': V('fps', label='oracle', gate=g, space=DIV_SPACE, photo=True)}


## Baselines (few-shot, self-training, FixMatch, Mean Teacher, active learning, previous HALT)

In [ ]:
# ===============================================================================
# BASELINES (same frozen CLIP features, same shots, same seeds, same episodic learner where a head is trained)
#   few-shot      : proto_raw (CLIP prototypes, no training) | linprobe (logistic regression) | cache_model (Tip-Adapter-style, image branch only)
#                   (the single-pass baseline itself is a Prototypical Network head)
#   semi-sup.     : selftrain   (Noisy-Student-style, one round: every pool image with conf >= PSEUDO_TAU gets its hard pseudo-label; capped)
#                   fixmatch    (FixMatch adapted to frozen features: weak = plain TTA view, strong = random cached augmented view; pseudo-label from the
#                                episode prototypes if conf >= tau_fm; CE on the strong view; trained jointly with the episodic labeled loss;
#                                unlabeled loss ramped up over the first 30% of episodes; tau_fm chosen per K on dev)
#                   meanteacher (Mean Teacher adapted to frozen features: EMA teacher head; MSE between student probs on one augmented view and teacher
#                                probs on another view of the same pool image; sigmoid ramp-up; evaluated with the EMA teacher; lambda chosen per K on dev)
#   active learn. : margin_al (oracle labels for the 4K lowest-margin pool samples) | ceal (oracle labels for the 2K most uncertain + capped pseudo-labels)
#   prior work    : halt_orig (re-implementation of the earlier multi-round HALT: confidence selection + oracle verification, 3 rounds)
# oracle-based methods report their number of expert queries; pseudo-label / semi-supervised methods use no extra labels.
# ===============================================================================
def _norm(x): return F.normalize(x, dim=-1)
def _cap(extra, conf):
    return [sorted(e, key=lambda i: -conf[i])[:PSEUDO_CAP] for e in extra]

def _train_semi(SF, seed, U_idx, mode, tau_fm=0.95, lambda_mt=10.0):
    """FixMatch / Mean Teacher on frozen features. SF [C, K, V, D] balanced labeled set; U_idx: train indices of the unlabeled subset."""
    set_seed(seed); n_ep = HP['n_episodes']; head = build_head(); opt, sched, loss_fn = _make_opt(head)
    C, K, V, D = SF.shape; Kc = min(K, HP['max_per_class_episode']); K_S, Q_S, QV = episode_plan(Kc); ar = torch.arange(C, device=device)[:, None]
    lab = torch.arange(C, device=device).repeat_interleave(Q_S + K_S * QV)
    Uw = TTA['train'][U_idx, 0]                                             # weak view  [N_u, D]
    Us = aug_views(U_idx, U_VIEWS)                                          # strong views [N_u, U_VIEWS, D]
    nU = len(U_idx); g = torch.Generator(device='cpu').manual_seed(seed)
    if mode == 'meanteacher':
        ema = build_head(); ema.load_state_dict(head.state_dict())
        for p in ema.parameters(): p.requires_grad = False
    n_mask = 0.0
    for ep in range(n_ep):
        head.train(); loss_l, protos, (x, xq) = _episode_fast(head, SF, Kc, K_S, QV, ar, lab, loss_fn)
        ub = torch.randint(0, nU, (U_BATCH,), generator=g).to(device); v1 = torch.randint(0, U_VIEWS, (U_BATCH,), generator=g).to(device)
        t = min(1.0, ep / max(1, SEMI_RAMP * n_ep)); ramp = math.exp(-5 * (1 - t) ** 2)
        xs = Us[ub, v1]
        if mode == 'fixmatch':
            with torch.no_grad():
                pw = torch.softmax(-torch.cdist(head(Uw[ub]), protos) ** 2, 1); cw, yh = pw.max(1); mask = (cw >= tau_fm).float()
            ce = F.cross_entropy(-torch.cdist(head(xs), protos) ** 2, yh, reduction='none'); loss_u = (ce * mask).mean(); wu = LAMBDA_U * ramp; n_mask += mask.mean().item()
        else:
            v2 = torch.randint(0, U_VIEWS, (U_BATCH,), generator=g).to(device)
            with torch.no_grad():
                ema.train(); te = ema(x.reshape(-1, D)).view(C, Kc, -1)[:, :K_S].mean(1)          # teacher prototypes from the SAME labeled support views
                pt = torch.softmax(-torch.cdist(ema(Us[ub, v2]), te) ** 2, 1)
            ps = torch.softmax(-torch.cdist(head(xs), protos) ** 2, 1); loss_u = ((ps - pt) ** 2).sum(1).mean()
            wu = lambda_mt * ramp
        loss = loss_l + wu * loss_u
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); sched.step()
        if mode == 'meanteacher':
            with torch.no_grad():
                for pe, ps_ in zip(ema.parameters(), head.parameters()): pe.mul_(EMA_DECAY).add_(ps_.detach(), alpha=1 - EMA_DECAY)
    out = ema if mode == 'meanteacher' else head; out.eval(); return out, n_mask / n_ep

def run_baseline(name, K, seed, semi_override=None):
    ctx = get_ctx(K, seed); t0 = tick(); y_ev = Y[EVAL_SPLIT]; Xe = TTA[EVAL_SPLIT]; shots = ctx['shots']; P0 = ctx['P0']; pool = ctx['pool']; yp = ctx['y_pool']; ov = OV_MASK[EVAL_SPLIT]
    row_extra = dict(n_queries=0)
    if name == 'proto_raw':
        pr = torch.stack([_norm(ctx['SFc'][c][:, 0]).mean(0) for c in range(N_CLS)]); e = _norm(Xe).mean(1)
        pred = (-torch.cdist(e, pr)).argmax(1).cpu().numpy(); row = metrics_from(y_ev, pred, ov)
    elif name in ('linprobe', 'cache_model'):
        if name == 'linprobe':
            Xtr = torch.cat([_norm(ctx['SFc'][c]).reshape(-1, FEAT_DIM) for c in range(N_CLS)]).cpu().numpy(); ytr = np.concatenate([[c] * (ctx['SFc'][c].shape[0] * ctx['SFc'][c].shape[1]) for c in range(N_CLS)])
            clf = LogisticRegression(max_iter=2000, C=1.0).fit(Xtr, ytr); pred = clf.predict(_norm(Xe).mean(1).cpu().numpy())
        else:
            keys = torch.cat([_norm(ctx['SFc'][c][:, 0]) for c in range(N_CLS)]); vals = F.one_hot(torch.cat([torch.full((ctx['SFc'][c].shape[0],), c) for c in range(N_CLS)]), N_CLS).float().to(device)
            aff = torch.exp(-5.0 * (1 - _norm(Xe).mean(1) @ keys.T)); pred = (aff @ vals).argmax(1).cpu().numpy()          # beta=5.0 fixed a priori
        row = metrics_from(y_ev, pred, ov)
    elif name in ('fixmatch', 'meanteacher'):
        U_idx = pool[np.random.default_rng(seed).choice(len(pool), size=min(U_SUB, len(pool)), replace=False)]
        sc = semi_override if semi_override is not None else SEMI_CFG[K]
        head, mask_rate = _train_semi(torch.stack(ctx['SFc']), seed, U_idx, name, sc['tau_fm'], sc['lambda_mt']); row = eval_head(head, class_protos(head, ctx['SFc']))
        row_extra.update(n_unlabeled=len(U_idx), fixmatch_mask_rate=mask_rate if name == 'fixmatch' else np.nan)
    else:
        extra = [[] for _ in range(N_CLS)]; pseudo = [[] for _ in range(N_CLS)]
        if name == 'selftrain':
            pred, conf = P0.argmax(1), P0.max(1)
            for i in np.where(conf >= PSEUDO_TAU)[0]: pseudo[int(pred[i])].append(int(i))
            pseudo = _cap(pseudo, conf)
        elif name == 'margin_al':
            s = np.sort(P0, 1); margin = s[:, -1] - s[:, -2]
            for i in np.argsort(margin)[:K * N_CLS]: extra[int(yp[i])].append(int(i))                                       # oracle reveals the true label
            row_extra['n_queries'] = K * N_CLS
        elif name == 'ceal':
            conf = P0.max(1); unc = np.argsort(conf)[:max(1, K * N_CLS // 2)]; q = set(unc.tolist())
            for i in unc: extra[int(yp[i])].append(int(i))                                                                  # queried: true label (NOT capped - v1 bug fixed)
            for i in np.where(conf >= PSEUDO_TAU)[0]:
                if int(i) not in q: pseudo[int(P0[i].argmax())].append(int(i))                                               # confident: pseudo-label
            pseudo = _cap(pseudo, conf); row_extra['n_queries'] = len(unc)
        elif name == 'halt_orig':
            cfg = V('hybrid', 1.0, 'oracle'); R = 3; sizes = [K // R + (1 if j < K % R else 0) for j in range(R)]; sizes = [s_ for s_ in sizes if s_ > 0]
            P, E = ctx['P0'], ctx['E0']; Es = ctx['Es0']; taken = np.zeros(len(yp), bool); info = dict(n_relaxed=0, n_forced=0, n_queries=0); rng = np.random.default_rng(seed)
            for j, r in enumerate(sizes):
                for c in range(N_CLS):
                    for i in pick_class(c, r, ctx, P, E, Es[c], taken, cfg, rng, info): extra[c].append(int(i)); taken[i] = True
                if j < len(sizes) - 1:
                    head, SFc, protos = retrain(ctx, extra, seed + j + 1); P, E = pool_state(ctx, head, protos)
                    Es = [F.normalize(head(SFc[c][:, 0]), dim=1).detach().cpu().numpy() for c in range(N_CLS)]                # refs in the CURRENT head space (v1 used stale refs)
            row_extra['n_queries'] = info['n_queries']
        extra = [e + p for e, p in zip(extra, pseudo)]
        if any(len(e) == 0 for e in extra): extra = [e if len(e) else [int(np.argmax(P0[:, c]))] for c, e in enumerate(extra)]
        head, SFc, protos = retrain(ctx, extra, seed); row = eval_head(head, protos)
        row['n_added'] = sum(len(e) for e in extra); row['pseudo_precision'] = float(np.mean([yp[i] == c for c in range(N_CLS) for i in extra[c]])) if row['n_added'] else np.nan
    row.update(row_extra)
    row['t_total'] = tick() - t0 + (ctx['t_train'] if name not in ('proto_raw', 'linprobe', 'cache_model', 'fixmatch', 'meanteacher') else 0); return row
BASELINES = ['proto_raw', 'linprobe', 'cache_model', 'selftrain', 'fixmatch', 'meanteacher', 'margin_al', 'ceal', 'halt_orig']


## Runner (config fingerprints) and statistics (pre-specified primary family + exploratory)

In [ ]:
# ===============================================================================
# RUNNER (resumable: every finished (K, seed, variant) is appended to a csv and skipped on restart)
# v2: every row stores a fingerprint of its configuration; a resumed run REFUSES to mix rows produced with different settings
# ===============================================================================
class WaitForShards(Exception): pass
def safe_csv(df, path):
    for k in range(20):                                             # write to a temp file, then rename (a reader never sees half a file); retry if Windows locks it
        try: df.to_csv(path + '.tmp', index=False); os.replace(path + '.tmp', path); return
        except PermissionError: time.sleep(0.5 + k)
    raise
def _read_one(p):
    for k in range(20):
        try: return pd.read_csv(p)
        except (PermissionError, pd.errors.EmptyDataError, pd.errors.ParserError): time.sleep(0.5 + k)
    return pd.read_csv(p)
def _read_all(csv_name):
    stem = csv_name[:-4]; fs = [f for f in os.listdir(OUT_DIR) if f == csv_name or re.fullmatch(re.escape(stem) + r'_shard\d+\.csv', f)]
    return pd.concat([_read_one(os.path.join(OUT_DIR, f)) for f in sorted(fs)], ignore_index=True) if fs else pd.DataFrame()
def fingerprint(K, name, cfg):
    d = dict(cfg=cfg, K=K, name=name, hp=HP, n_aug=N_AUG, val=(VAL_MODE, VAL_PHOTO_FRAC, VAL_SPLIT_SEED), split=EVAL_SPLIT,
             semi=dict(U_SUB=U_SUB, U_VIEWS=U_VIEWS, U_BATCH=U_BATCH, LAMBDA_U=LAMBDA_U, EMA=EMA_DECAY, RAMP=SEMI_RAMP, CAP=PSEUDO_CAP, TAU=PSEUDO_TAU,
                       cfg=(cfg if isinstance(cfg, dict) else SEMI_CFG.get(K))) if (name in BASELINES or str(name).startswith(('fm=', 'mt='))) else None)
    return hashlib.md5(json.dumps(d, sort_keys=True, default=str).encode()).hexdigest()[:10]
def run_grid(names, Ks, seeds, csv_name, cfg_fn=None):
    """runs this shard's seeds (seeds[SHARD_ID::N_SHARDS]); cfg_fn(K) -> {name: cfg}; returns the merged table of ALL shards"""
    cfg_fn = cfg_fn or variants
    my_csv = csv_name if N_SHARDS == 1 else csv_name[:-4] + f'_shard{SHARD_ID}.csv'; my_path = os.path.join(OUT_DIR, my_csv)
    allp = _read_all(csv_name); done = {}
    if len(allp): done = {(int(a), int(b), c): d for a, b, c, d in zip(allp.K, allp.seed, allp.variant, allp.get('fp', pd.Series([None] * len(allp))))}
    rows = pd.read_csv(my_path).to_dict('records') if os.path.exists(my_path) else []; t0 = time.time(); my_seeds = list(seeds)[SHARD_ID::N_SHARDS]
    for K in Ks:
        cfgs = cfg_fn(K)
        for s in my_seeds:
            ran = False
            for n in names:
                fp = fingerprint(K, n, cfgs.get(n) if n not in BASELINES else n)
                if (K, s, n) in done:
                    if done[(K, s, n)] != fp: raise RuntimeError(f'{csv_name}: row (K={K}, seed={s}, {n}) was produced with a DIFFERENT configuration. Delete that csv or change RUN_TAG.')
                    continue
                if n in BASELINES: r = run_baseline(n, K, s)
                elif isinstance(cfgs.get(n), dict) and 'baseline' in cfgs[n]: r = run_baseline(cfgs[n]['baseline'], K, s, cfgs[n]['semi'])
                else: r = run_variant(K, s, n, cfgs[n])
                r.update(K=K, seed=s, variant=n, stage=STAGE, eval_split=EVAL_SPLIT, fp=fp, cfg=json.dumps(cfgs.get(n), default=str) if n not in BASELINES else n); rows.append(r); ran = True
            if not ran: continue
            safe_csv(pd.DataFrame(rows), my_path); print(f"  [{csv_name}] shard {SHARD_ID}/{N_SHARDS}  K={K} seed={s} done  ({(time.time()-t0)/60:.1f} min)")
    if WAIT_SHARDS and N_SHARDS > 1:                               # barrier: wait until every shard has finished this grid
        need = {(K, s, n) for K in Ks for s in seeds for n in names}; t1 = time.time(); last = 0
        while True:
            allp = _read_all(csv_name); have = set(zip(allp.K.astype(int), allp.seed.astype(int), allp.variant)) if len(allp) else set(); miss = len(need - have)
            if not miss: break
            if time.time() - last > 300: print(f'  [{csv_name}] shard {SHARD_ID}: waiting for the other shards ({miss} runs missing, {(time.time() - t1) / 60:.0f} min)'); last = time.time()
            if time.time() - t1 > 12 * 3600: raise WaitForShards(f'{csv_name}: other shards did not finish within 12 h')
            time.sleep(15)
    return _read_all(csv_name)
def require_complete(df, names, Ks, seeds):
    need = {(K, s, n) for K in Ks for s in seeds for n in names}; have = set(zip(df.K, df.seed, df.variant)) if len(df) else set()
    if need - have: raise WaitForShards(f'{len(need - have)} of {len(need)} runs are still missing (other shards are still running). Re-run this cell when every shard has finished.')

# ---- statistics ---------------------------------------------------------------------------------------------------------------------
def tci(x):
    x = np.asarray(x, float); return stats.t.interval(0.95, len(x) - 1, loc=x.mean(), scale=stats.sem(x)) if len(x) > 1 and x.std() > 0 else (x.mean(), x.mean())
def summarize(df, metric='acc'):
    rows = []
    for (K, v), g in df.groupby(['K', 'variant']):
        lo, hi = tci(g[metric]); rows.append(dict(K=K, variant=v, n=len(g), mean=100 * g[metric].mean(), sd=100 * g[metric].std(ddof=1), ci_lo=100 * lo, ci_hi=100 * hi, median=100 * g[metric].median()))
    return pd.DataFrame(rows)
def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); m = len(p); adj = np.empty(m); run = 0
    for rank, i in enumerate(o): run = max(run, (m - rank) * p[i]); adj[i] = min(1, run)
    return adj
def paired_tests(df, pairs, metric='acc', family=None):
    """per-K paired tests. family: name of the Holm family ('primary' = the pre-specified list; anything else = exploratory, unadjusted)"""
    rows = []
    for K in sorted(df.K.unique()):
        for a, b in pairs:
            A = df[(df.K == K) & (df.variant == a)].set_index('seed')[metric]; B = df[(df.K == K) & (df.variant == b)].set_index('seed')[metric]; ix = A.index.intersection(B.index)
            if len(ix) < 3: continue
            d = (A[ix] - B[ix]).values; sd = d.std(ddof=1)
            try: pw = stats.wilcoxon(d).pvalue if np.any(d != 0) else 1.0
            except ValueError: pw = np.nan
            pt = stats.ttest_rel(A[ix], B[ix]).pvalue if sd > 0 else (1.0 if d.mean() == 0 else 0.0)
            lo, hi = tci(d); rows.append(dict(K=K, A=a, B=b, n=len(ix), mean_diff_pp=100 * d.mean(), ci_lo=100 * lo, ci_hi=100 * hi, p_ttest=pt, p_wilcoxon=pw,
                                              cohen_dz=d.mean() / sd if sd > 0 else np.nan, wins=int((d > 0).sum()), ties=int((d == 0).sum())))
    out = pd.DataFrame(rows)
    if len(out):
        out['family'] = family or 'exploratory'
        if family == 'primary': out['p_wilcoxon_holm'] = holm(out['p_wilcoxon'].fillna(1)); out['p_ttest_holm'] = holm(out['p_ttest'].fillna(1))
    return out
def primary_pairs(new): return [(new, 'baseline'), (new, 'random_pseudo'), (new, 'conf_pseudo')]   # x 4 K = 12 pre-specified comparisons (Holm over these 12 only)


## E2 - tuning on validation only (dev stage)

In [ ]:
# ===============================================================================
# E2  TUNING ON VALIDATION ONLY (dev stage).  Everything chosen here is written to frozen_config.json; the final stage only loads it.
#   (i)  original hybrid: w (0..1, step 0.1) then tau (0.5..0.9), re-tuned on the photo-level validation split as in v1; per-K curves with SDs (R1C4, R4-8, R6-5, R6-6)
#   (ii) NEW family: hard gate tau_gate(K) (tuned with gate_fps_photo) and the rank weight w(K) of norm_hybrid, each chosen PER K
#   (iii) FixMatch tau_fm(K) and Mean Teacher lambda(K): one knob each, per K - the same tuning budget as the proposed method
# ===============================================================================
def per_k_choice(df, col):
    """per K: the value with the highest mean validation accuracy over the dev seeds (all values share the same seeds/shots -> paired comparison)"""
    t = df.groupby(['K', col]).acc.agg(['mean', 'std']).reset_index(); best = t.loc[t.groupby('K')['mean'].idxmax()]
    return {int(k): float(v) for k, v in zip(best.K, best[col])}, t

if STAGE == 'dev':
    # ---- (i) ORIGINAL hybrid, re-tuned on the photo-level validation split exactly as in v1: weights at tau=0.9 (no adaptive), then tau at w* (adaptive on) ----
    #      one value of w and of tau for all K, as in the original method (pooled over K); per-K curves with seed SDs are plotted (R1C4, R4-8, R6-5, R6-6)
    wn_ = [f'w={w}' for w in SWEEP_WEIGHTS]
    SW = run_grid(wn_, K_LIST, SEEDS, 'sweep_weight.csv', cfg_fn=lambda K: {f'w={w}': V('hybrid', w) for w in SWEEP_WEIGHTS})
    require_complete(SW, wn_, K_LIST, SEEDS); SW['w'] = SW['variant'].str[2:].astype(float)
    W_CONF = float(SW.groupby('w').acc.mean().idxmax())
    print(f'weight sweep at tau={TAU} (validation acc %, per K and pooled):\n', (100 * SW.groupby(['K', 'w']).acc.mean()).unstack('w').round(2), '\n pooled:', (100 * SW.groupby('w').acc.mean()).round(2).to_dict(), f'\n-> w* = {W_CONF}')
    tn_ = [f'tau={t}' for t in SWEEP_TAUS]
    ST = run_grid(tn_, K_LIST, SEEDS, 'sweep_tau.csv', cfg_fn=lambda K: {f'tau={t}': dict(V('hybrid', W_CONF, adaptive=True), tau=t) for t in SWEEP_TAUS})
    require_complete(ST, tn_, K_LIST, SEEDS); ST['tau'] = ST['variant'].str[4:].astype(float)
    TAU = float(ST.groupby('tau').acc.mean().idxmax())
    print(f'tau sweep at w={W_CONF} (validation acc %, per K and pooled):\n', (100 * ST.groupby(['K', 'tau']).acc.mean()).unstack('tau').round(2), '\n pooled:', (100 * ST.groupby('tau').acc.mean()).round(2).to_dict(), f'\n-> tau* = {TAU}')
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
    for a_, (d, col, lab) in zip(ax, [(SW, 'w', 'confidence weight w (dissimilarity weight = 1-w)'), (ST, 'tau', r'confidence threshold $\tau$')]):
        for K, g_ in d.groupby('K'): s_ = g_.groupby(col).acc.agg(['mean', 'std']); a_.errorbar(s_.index, 100 * s_['mean'], 100 * s_['std'], marker='o', capsize=3, label=f'{K}-shot')
        a_.set_xlabel(lab); a_.set_ylabel('validation accuracy (%)'); a_.legend(); a_.grid(alpha=.3)
    plt.tight_layout(); IS_MAIN and plt.savefig(os.path.join(OUT_DIR, 'fig_sweeps_original_hybrid.png'), dpi=200); plt.show()

    # ---- (ii) NEW family sweeps ---------------------------------------------------------------------------------------------------------------------
    gnames = [f'g={g}' for g in TAU_GATE_GRID]
    SG = run_grid(gnames, K_LIST, SEEDS, 'sweep_gate.csv', cfg_fn=lambda K: {f'g={g}': V('fps', gate=g, space=DIV_SPACE, photo=True) for g in TAU_GATE_GRID})
    require_complete(SG, gnames, K_LIST, SEEDS); SG['g'] = SG['variant'].str[2:].astype(float)
    TAU_GATE, tg = per_k_choice(SG, 'g'); print('gate sweep (gate_fps_photo, validation acc %):\n', (100 * tg.pivot(index='K', columns='g', values='mean')).round(2), '\n-> tau_gate(K) =', TAU_GATE)
    wnames = [f'w={w}' for w in W_NORM_GRID]
    SWN = run_grid(wnames, K_LIST, SEEDS, 'sweep_wnorm.csv', cfg_fn=lambda K: {f'w={w}': V('norm', w, gate=NORM_GATE, space=DIV_SPACE) for w in W_NORM_GRID})
    require_complete(SWN, wnames, K_LIST, SEEDS); SWN['wn'] = SWN['variant'].str[2:].astype(float)
    W_NORM, tw = per_k_choice(SWN, 'wn'); print('rank-weight sweep (norm_hybrid, validation acc %):\n', (100 * tw.pivot(index='K', columns='wn', values='mean')).round(2), '\n-> w_norm(K) =', W_NORM)
    print('pseudo-label precision / gate relaxations in the gate sweep:\n', SG.groupby(['K', 'g'])[['pseudo_precision', 'n_relaxed', 'div_pairwise']].mean().round(3).unstack('g'))
    # ---- (iii) semi-supervised baselines: one knob each, per K, same budget as the proposed method (fairness) ----------------------------------------
    sn = [f'fm={t}' for t in TAU_FM_GRID] + [f'mt={l}' for l in LAMBDA_MT_GRID]
    SS = run_grid(sn, K_LIST, SEEDS, 'sweep_semi.csv', cfg_fn=lambda K: {**{f'fm={t}': {'baseline': 'fixmatch', 'semi': dict(tau_fm=t, lambda_mt=SEMI_CFG[K]['lambda_mt'])} for t in TAU_FM_GRID},
                                                                         **{f'mt={l}': {'baseline': 'meanteacher', 'semi': dict(tau_fm=SEMI_CFG[K]['tau_fm'], lambda_mt=l)} for l in LAMBDA_MT_GRID}})
    require_complete(SS, sn, K_LIST, SEEDS); SS['kind'] = SS['variant'].str[:2]; SS['val_'] = SS['variant'].str[3:].astype(float)
    best_fm, tfm = per_k_choice(SS[SS.kind == 'fm'], 'val_'); best_mt, tmt = per_k_choice(SS[SS.kind == 'mt'], 'val_')
    SEMI_CFG = {K: dict(tau_fm=best_fm[K], lambda_mt=best_mt[K]) for K in K_LIST}
    print('FixMatch tau_fm sweep (validation acc %):\n', (100 * tfm.pivot(index='K', columns='val_', values='mean')).round(2))
    print('Mean Teacher lambda sweep (validation acc %):\n', (100 * tmt.pivot(index='K', columns='val_', values='mean')).round(2), '\n-> SEMI_CFG =', SEMI_CFG)
    FROZEN = {'TAU': TAU, 'W_CONF': W_CONF, 'TAU_GATE': TAU_GATE, 'W_NORM': W_NORM, 'SEMI_CFG': SEMI_CFG, 'DIV_SPACE': DIV_SPACE, 'NORM_GATE': NORM_GATE, 'PSEUDO_TAU': PSEUDO_TAU,
              'VAL_MODE': VAL_MODE, 'VAL_PHOTO_FRAC': VAL_PHOTO_FRAC, 'VAL_SPLIT_SEED': VAL_SPLIT_SEED, 'VAL_SPLIT_INFO': VAL_SPLIT_INFO, 'weights_grid': SWEEP_WEIGHTS, 'tau_grid': SWEEP_TAUS,
              'chosen_on': 'validation (dev seeds)', 'gate_grid': TAU_GATE_GRID, 'w_norm_grid': W_NORM_GRID, 'dev_seeds': SEEDS, 'NEW_METHOD': None,
              'note': 'NEW selection family designed after inspecting v1/v2 dev results on the OFFICIAL (patch-level, leaky) validation split; v3 re-tunes everything on a '
                      'photo-level validation split. The test set was never used for any choice.'}
    if IS_MAIN: json.dump(FROZEN, open(FROZEN_JSON, 'w'), indent=2)
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
    for a, (d, col, lab) in zip(ax, [(SG, 'g', r'confidence gate $\tau_g$ (gate_fps_photo)'), (SWN, 'wn', 'rank weight of confidence (norm_hybrid)')]):
        for K, g_ in d.groupby('K'): s_ = g_.groupby(col).acc.agg(['mean', 'std']); a.errorbar(s_.index, 100 * s_['mean'], 100 * s_['std'], marker='o', capsize=3, label=f'{K}-shot')
        a.set_xlabel(lab); a.set_ylabel('validation accuracy (%)'); a.legend(); a.grid(alpha=.3)
    plt.tight_layout(); IS_MAIN and plt.savefig(os.path.join(OUT_DIR, 'fig_sweeps_new_family.png'), dpi=200); plt.show()
else:
    print('final stage: all knobs FROZEN from the dev stage ->', dict(TAU=TAU, W_CONF=W_CONF, TAU_GATE=TAU_GATE, W_NORM=W_NORM, SEMI_CFG=SEMI_CFG, NEW_METHOD=NEW_METHOD), '(no tuning here)')


## E1 / E3 / E4 - main results, ablations, baselines, decision rule

In [ ]:
# ===============================================================================
# E1 + E3 + E4  MAIN RESULTS, ABLATIONS, BASELINES  (+ the pre-specified method decision at the end of the dev stage)
# ===============================================================================
MAIN_NAMES = list(variants(K_LIST[0]).keys())
RES = run_grid(MAIN_NAMES, K_LIST, SEEDS, 'results_main.csv'); require_complete(RES, MAIN_NAMES, K_LIST, SEEDS)
BRES = run_grid(BASELINES, K_LIST, SEEDS, 'results_baselines.csv'); require_complete(BRES, BASELINES, K_LIST, SEEDS)
ALL = pd.concat([RES, BRES], ignore_index=True); ALLSUM = summarize(ALL); IS_MAIN and ALLSUM.to_csv(os.path.join(OUT_DIR, 'summary_all_methods.csv'), index=False)
SUM = ALLSUM
print(f'accuracy (%) on {EVAL_SPLIT}, mean over {len(SEEDS)} seeds:\n', ALLSUM.pivot(index='variant', columns='K', values='mean').round(2).reindex(MAIN_NAMES + BASELINES))

# ---- decision rule (pre-specified, DEV only): which candidate goes to the test stage ---------------------------------------------------------------
def mean_diff(df, a, b, K):
    A = df[(df.K == K) & (df.variant == a)].set_index('seed').acc; B = df[(df.K == K) & (df.variant == b)].set_index('seed').acc; ix = A.index.intersection(B.index)
    return 100 * float((A[ix] - B[ix]).mean())
if STAGE == 'dev':
    dec = []
    for i, c in enumerate(CANDIDATES):
        db = [mean_diff(RES, c, 'baseline', K) for K in K_LIST]; dr = [mean_diff(RES, c, 'random_pseudo', K) for K in K_LIST]
        dec.append(dict(candidate=c, order=i, **{f'vs_base_K{K}': v for K, v in zip(K_LIST, db)}, **{f'vs_random_K{K}': v for K, v in zip(K_LIST, dr)},
                        mean_vs_random=float(np.mean(dr)), mean_vs_base=float(np.mean(db)), eligible=bool(min(db) >= -NONINF_MARGIN_PP)))
    DEC = pd.DataFrame(dec)
    if all(W_NORM[K] == 1.0 for K in K_LIST): DEC.loc[DEC.candidate == 'norm_hybrid_pseudo', 'eligible'] = False; print('norm_hybrid has w_norm = 1 at every K: it is pure confidence ranking (= conf_pseudo) and is not a new method -> excluded')
    pool_ = DEC[DEC.eligible] if DEC.eligible.any() else DEC
    top = pool_.mean_vs_random.max(); NEW_METHOD = pool_[pool_.mean_vs_random >= top - 0.10].sort_values('order').candidate.iloc[0]   # ties (0.1 pp) -> simpler method
    print('decision table (dev, pp):\n', DEC.round(2).to_string()); print(f'-> NEW_METHOD by the rule on this backbone = {NEW_METHOD}')
    OWN_CHOICE = NEW_METHOD
    if NEW_METHOD_FIXED:                       # second backbone: the method is NOT re-chosen; the rule's own choice is only recorded
        NEW_METHOD = NEW_METHOD_FIXED; print(f'-> NEW_METHOD fixed to the ECU-MSS + CLIP choice: {NEW_METHOD} (the rule would have picked {OWN_CHOICE} on {BB_LABEL})')
    FROZEN.update(NEW_METHOD=NEW_METHOD, own_rule_choice=OWN_CHOICE, method_fixed_from=(REF_FROZEN_JSON if NEW_METHOD_FIXED else None), backbone=BB_LABEL, dataset=DATASET, decision_table=DEC.round(3).to_dict('records'),
        decision_rule=f'eligible if mean paired diff vs baseline >= -{NONINF_MARGIN_PP} pp at every K; among eligible maximise the mean over K of the paired diff vs random_pseudo; '
                      'ties within 0.1 pp -> simpler method (order of CANDIDATES); if none eligible, best by the same criterion (flagged)',
        any_eligible=bool(DEC.eligible.any()), new_beats_random_on_dev=bool(DEC.set_index('candidate').loc[NEW_METHOD, 'mean_vs_random'] > 0),
        new_beats_baseline_on_dev=bool(DEC.set_index('candidate').loc[NEW_METHOD, 'mean_vs_base'] > 0))
    if IS_MAIN: json.dump(FROZEN, open(FROZEN_JSON, 'w'), indent=2); DEC.to_csv(os.path.join(OUT_DIR, 'decision_table.csv'), index=False)

# ---- statistics: 12 pre-specified primary comparisons (Holm) + exploratory comparisons (unadjusted, labelled as such) -------------------------------
PT_PRIMARY = paired_tests(ALL, primary_pairs(NEW_METHOD), family='primary')
EXPL = [('hybrid_pseudo', 'baseline'), ('hybrid_pseudo', 'random_pseudo'), ('hybrid_pseudo', 'conf_pseudo'), ('hybrid_adapt_pseudo', 'hybrid_pseudo'),
        ('dissim_pseudo', 'baseline'), ('conf_pseudo', 'baseline'), ('random_pseudo', 'baseline'), ('random_gate_pseudo', 'random_pseudo'),
        (NEW_METHOD, 'random_gate_pseudo'), (NEW_METHOD, 'hybrid_pseudo'), (NEW_METHOD, 'selftrain'), (NEW_METHOD, 'fixmatch'), (NEW_METHOD, 'meanteacher'),
        ('gate_fps_photo_oracle', 'random_correct_oracle'), ('gate_fps_photo_oracle', 'random_oracle'), ('hybrid_oracle', 'random_correct_oracle'),
        ('conf_oracle', 'random_correct_oracle'), ('gate_fps_photo_oracle', 'margin_al'), ('gate_fps_photo_oracle', 'halt_orig'), ('gate_fps_photo_oracle', 'gate_fps_photo_pseudo')]
EXPL = [p for p in dict.fromkeys(EXPL) if p[0] != p[1]]
PT_EXPL = paired_tests(ALL, EXPL, family='exploratory')
PT = pd.concat([PT_PRIMARY, PT_EXPL], ignore_index=True); IS_MAIN and PT.to_csv(os.path.join(OUT_DIR, 'paired_tests_acc.csv'), index=False)
print(f'PRIMARY comparisons (NEW = {NEW_METHOD}; Holm over these {len(PT_PRIMARY)}):\n', PT_PRIMARY.round(4).to_string())
print('EXPLORATORY comparisons (unadjusted):\n', PT_EXPL[['K', 'A', 'B', 'mean_diff_pp', 'ci_lo', 'ci_hi', 'p_wilcoxon', 'wins']].round(3).to_string())


## E5 / E6 - cost, diagnostics, LaTeX tables, figures

In [ ]:
# MAIN_SHARD_ONLY
# ===============================================================================
# E5 cost, E6 diagnostics, paper tables (LaTeX) and figures
# ===============================================================================
dev_name = (torch.cuda.get_device_name(0) if device.type == 'cuda' else 'CPU'); print('hardware:', dev_name)
NEW = NEW_METHOD; ORACLE_NEW = 'gate_fps_photo_oracle'
# ---- E5: wall-clock per stage (s, mean over seeds); total = teacher training + teacher inference on the pool + selection + adaptive + student training --------
tv = RES[RES.variant.isin(['baseline', 'hybrid_adapt_pseudo', 'hybrid_pseudo', NEW])].copy()
tv.loc[tv.variant == 'baseline', 't_pool'] = 0.0                                   # the baseline does not need the pool
tm = tv.groupby(['K', 'variant'])[['t_train', 't_pool', 't_select', 't_adapt', 't_final']].mean()
tm['total_s'] = tm.sum(1); tm = tm.round(2); print(tm)
ov = tm['total_s'].unstack('variant'); ov['overhead_x_new'] = ov[NEW] / ov['baseline']; print('overhead vs single-pass baseline:\n', ov.round(2)); tm.to_csv(os.path.join(OUT_DIR, 'timing_per_stage.csv'))
print('adaptive retraining (ablation) stop reasons:\n', RES[RES.variant == 'hybrid_adapt_pseudo'].groupby(['K', 'adapt_stop']).size().unstack(fill_value=0))
print('mean pool confidence at start:', RES[RES.variant == 'hybrid_pseudo'].groupby('K').mean_conf_start.mean().round(3).to_dict(), '| tau =', TAU)

# ---- E6: pool size / balance, pseudo-label quality, diversity, verification queries, split-overlap accuracy ------------------------------------------
for K in K_LIST:
    c = get_ctx(K, SEEDS[0]); print(f"K={K}: pool size {len(c['pool'])} | pool class counts {np.bincount(c['y_pool'], minlength=N_CLS).tolist()}")
diag_cols = ['pseudo_precision'] + [f'pseudo_prec_{n}' for n in CLASSES] + ['div_pairwise', 'div_to_shots', 'same_photo_rate', 'n_relaxed', 'n_forced', 'n_queries']
DQ = RES[RES.variant != 'baseline'].groupby(['K', 'variant'])[diag_cols].mean().round(3)
print('selected-sample diagnostics (true labels used for these LOGS only):\n', DQ); DQ.to_csv(os.path.join(OUT_DIR, 'selection_diagnostics.csv'))
QQ = ALL[ALL.n_queries > 0].groupby(['K', 'variant']).n_queries.mean().unstack('K').round(1); print('expert queries (verification or labeling) per run:\n', QQ); QQ.to_csv(os.path.join(OUT_DIR, 'queries.csv'))
OVA = ALL[ALL.variant.isin(['baseline', NEW, 'random_pseudo', 'conf_pseudo'])].groupby(['K', 'variant'])[['acc', 'acc_ov', 'acc_nov']].mean().mul(100).round(2)
print(f'accuracy on {EVAL_SPLIT} patches whose photo key occurs in train (acc_ov) vs the rest (acc_nov):\n', OVA); OVA.to_csv(os.path.join(OUT_DIR, 'overlap_accuracy.csv'))
print(f"embedding facts: {BB_LABEL} image feature {FEAT_DIM}-d (frozen) -> head {FEAT_DIM}->{HP['hidden_dim']}->{HP['output_dim']}; classification = squared Euclidean distance to class prototypes in head space; "
      f"original hybrid: dissimilarity = 1 - cosine on L2-normalised head embeddings (plain view); NEW family: 1 - cosine on L2-normalised frozen {BB_LABEL} embeddings (mean of 3 TTA views); "
      f"hard pseudo-labels; K per class by construction.")

# ---- LaTeX ---------------------------------------------------------------------------------------------------------------------------------------------
def acc_of(K, v, metric='acc'):
    return ALL[(ALL.K == K) & (ALL.variant == v)].set_index('seed')[metric] * 100
def ms(K, v, metric='acc'):
    g = acc_of(K, v, metric); return f"{g.mean():.2f} $\\pm$ {g.std(ddof=1):.2f}" if len(g) else '--'
def dd(K, a, b, P=None, metric='acc'):
    A, B = acc_of(K, a, metric), acc_of(K, b, metric); ix = A.index.intersection(B.index); d = (A[ix] - B[ix]).mean(); star = ''
    if P is not None and len(P):
        r = P[(P.K == K) & (P.A == a) & (P.B == b)]
        if len(r) and 'p_wilcoxon_holm' in r and r.p_wilcoxon_holm.iloc[0] < 0.05: star = '$^{*}$'
    return f"{d:+.2f}{star}"
def latex_table(cols, diffs, caption, label, extra_fmt=None, metric='acc', P=None):
    P = PT_PRIMARY if P is None else P; mname = 'Accuracy' if metric == 'acc' else 'Macro-F1'
    hdr = ' & '.join(['Setting'] + [c[0] for c in cols] + [d[0] for d in diffs])
    L = [r'\begin{table}[!htbp]', r'\centering', rf'\caption{{{caption}}}', rf'\label{{{label}}}', r'\scriptsize', r'\setlength{\tabcolsep}{3pt}',
         r'\begin{tabular}{@{}l' + 'c' * (len(cols) + len(diffs)) + r'@{}}', r'\toprule',
         (rf' & \multicolumn{{{len(cols)}}}{{c}}{{{mname} (\%, mean $\pm$ SD)}} & \multicolumn{{{len(diffs)}}}{{c}}{{Paired difference (pp)}} \\' if diffs else ''),
         (rf'\cmidrule(lr){{2-{len(cols) + 1}}}\cmidrule(lr){{{len(cols) + 2}-{len(cols) + len(diffs) + 1}}}' if diffs else ''), hdr + r' \\', r'\midrule']
    for K in K_LIST: L.append(' & '.join([f'{K}-shot'] + [ms(K, c[1], metric) for c in cols] + [dd(K, d[1], d[2], P, metric) for d in diffs]) + r' \\')
    L += [r'\bottomrule', r'\end{tabular}', r'\end{table}']; return '\n'.join([l for l in L if l])
nseed = f'{"ECU-MSS" if DATASET == "ecu" else "DeepSeagrass"}, {BB_LABEL}, {len(SEEDS)} seeds, {"validation" if EVAL_SPLIT == "val" else "test"} set'
tex = [latex_table([('Baseline', 'baseline'), ('Random', 'random_pseudo'), ('Conf', 'conf_pseudo'), ('Dissim', 'dissim_pseudo'), ('Hybrid', 'hybrid_pseudo'), ('Proposed', NEW)],
                   [(r'vs.\ Base', NEW, 'baseline'), (r'vs.\ Rand', NEW, 'random_pseudo'), (r'vs.\ Conf', NEW, 'conf_pseudo')],
                   f'Leakage-free protocol ({nseed}). Proposed = {NEW.replace("_", r"\_")}; Hybrid = original score (w={W_CONF}, $\\tau$={TAU}). '
                   r'Differences are paired over seeds; $^{*}$: Holm-adjusted Wilcoxon $p<0.05$ within the 12 pre-specified primary comparisons.', 'tab:main_pseudo'),
       latex_table([('Random labels', 'random_oracle'), ('Random verified', 'random_correct_oracle'), ('Conf', 'conf_oracle'), ('Hybrid', 'hybrid_oracle'), ('Gate+FPS+photo', ORACLE_NEW), ('Margin AL', 'margin_al'), ('Prev. HALT', 'halt_orig')],
                   [], f'Verification-based active learning (oracle-assisted; an expert confirms or rejects the predicted class) ({nseed}). Expert queries per run are listed in Table~\\ref{{tab:queries}}.', 'tab:main_oracle'),
       latex_table([('ProtoNet head', 'baseline'), ('Raw backbone proto.', 'proto_raw'), ('Linear probe', 'linprobe'), ('Tip-Adapter', 'cache_model'), ('Self-training', 'selftrain'), ('FixMatch', 'fixmatch'), ('Mean Teacher', 'meanteacher'), ('Proposed', NEW)],
                   [], f'Few-shot and semi-supervised baselines on the same frozen {BB_LABEL} features, shots and seeds ({nseed}).', 'tab:baselines')]
QL = [r'\begin{table}[!htbp]', r'\centering', r'\caption{Expert queries per run (mean over seeds).}', r'\label{tab:queries}', r'\footnotesize', r'\begin{tabular}{@{}l' + 'c' * len(K_LIST) + r'@{}}', r'\toprule',
      'Method & ' + ' & '.join(f'{K}-shot' for K in K_LIST) + r' \\', r'\midrule'] + \
     [f"{v.replace('_', r'\_')} & " + ' & '.join(f'{QQ.loc[v, K]:.1f}' if K in QQ.columns else '--' for K in K_LIST) + r' \\' for v in QQ.index] + [r'\bottomrule', r'\end{tabular}', r'\end{table}']
TL = [r'\begin{table}[!htbp]', r'\centering', rf'\caption{{Wall-clock time per run (s, mean over seeds) on {dev_name}; {BB_LABEL} feature extraction is a one-off cache shared by all methods.}}', r'\label{tab:timing}', r'\footnotesize',
      r'\begin{tabular}{@{}llrrrrrr@{}}', r'\toprule', r'K & Method & Teacher & Pool inference & Selection & Adaptive & Student & Total \\', r'\midrule'] + \
     [f"{K} & {v.replace('_', r'\_')} & " + ' & '.join(f'{x:.2f}' for x in tm.loc[(K, v)].values) + r' \\' for (K, v) in tm.index] + [r'\bottomrule', r'\end{tabular}', r'\end{table}']
tex += ['\n'.join(QL), '\n'.join(TL)]

# ---- v4 extra tables: macro-F1 (secondary; important for the imbalanced DeepSeagrass test set), per-class F1, teacher calibration, data splits ----------------
PT_F1 = paired_tests(ALL, primary_pairs(NEW), metric='macro_f1', family='secondary')
if len(PT_F1): PT_F1['p_wilcoxon_holm'] = holm(PT_F1['p_wilcoxon'].fillna(1)); PT_F1.to_csv(os.path.join(OUT_DIR, 'paired_tests_macro_f1.csv'), index=False)
tex.append(latex_table([('Baseline', 'baseline'), ('Random', 'random_pseudo'), ('Conf', 'conf_pseudo'), ('Proposed', NEW), ('Self-training', 'selftrain'), ('FixMatch', 'fixmatch'), ('Mean Teacher', 'meanteacher')],
                       [(r'vs.\ Base', NEW, 'baseline'), (r'vs.\ Rand', NEW, 'random_pseudo'), (r'vs.\ Conf', NEW, 'conf_pseudo')],
                       f'Macro-F1 ({nseed}); secondary outcome, Holm-corrected separately over the same 12 comparisons ($^{{*}}$: $p<0.05$).', 'tab:macro_f1', metric='macro_f1', P=PT_F1))
PC = [r'\begin{table}[!htbp]', r'\centering', rf'\caption{{Per-class F1 (\%, mean $\pm$ SD over seeds; {nseed}).}}', r'\label{tab:per_class}', r'\scriptsize', r'\setlength{\tabcolsep}{3pt}',
      r'\begin{tabular}{@{}ll' + 'c' * N_CLS + r'@{}}', r'\toprule', 'Setting & Method & ' + ' & '.join(CLASSES) + r' \\', r'\midrule']
for K in K_LIST:
    for v, lab in [('baseline', 'Baseline'), ('random_pseudo', 'Random'), ('conf_pseudo', 'Conf'), (NEW, 'Proposed'), ('meanteacher', 'Mean Teacher')]:
        PC.append(f'{K}-shot & {lab} & ' + ' & '.join(ms(K, v, f'f1_{n}') for n in CLASSES) + r' \\')
    PC.append(r'\midrule' if K != K_LIST[-1] else '')
PC += [r'\bottomrule', r'\end{tabular}', r'\end{table}']; tex.append('\n'.join([l for l in PC if l]))
CAL = RES[RES.variant == 'baseline'].groupby('K')[['teacher_pool_acc', 'teacher_pool_conf', 'teacher_pool_ece', 'teacher_acc_conf_ge_0_9', 'teacher_frac_conf_ge_0_9']].agg(['mean', 'std'])
print('teacher calibration on the unlabeled pool (ground truth used only for this diagnostic):\n', CAL.round(3)); CAL.to_csv(os.path.join(OUT_DIR, 'teacher_calibration.csv'))
CL = [r'\begin{table}[!htbp]', r'\centering', rf'\caption{{Teacher on the unlabeled pool (mean $\pm$ SD over seeds; {nseed}): accuracy, mean confidence, expected calibration error (15 bins), and accuracy and share of pool images with confidence $\geq 0.9$.}}',
      r'\label{tab:calibration}', r'\footnotesize', r'\begin{tabular}{@{}lccccc@{}}', r'\toprule', r'Setting & Accuracy & Mean conf. & ECE & Acc.\ (conf $\geq$ 0.9) & Share conf $\geq$ 0.9 \\', r'\midrule'] + \
     [f'{K}-shot & ' + ' & '.join(f"{100 * CAL.loc[K, (c, 'mean')]:.1f} $\\pm$ {100 * CAL.loc[K, (c, 'std')]:.1f}" for c in ['teacher_pool_acc', 'teacher_pool_conf', 'teacher_pool_ece', 'teacher_acc_conf_ge_0_9', 'teacher_frac_conf_ge_0_9']) + r' \\' for K in K_LIST] + \
     [r'\bottomrule', r'\end{tabular}', r'\end{table}']
tex.append('\n'.join(CL))
_pool_n = {K: len(get_ctx(K, SEEDS[0])['pool']) for K in K_LIST}
DS_rows = [('Shots and pool (training)', Y['train'][TRAIN_ELIGIBLE], {PHOTO_KEY['train'][i] for i in np.where(TRAIN_ELIGIBLE)[0]}),
           ('Validation (model selection)' if DATASET == 'ecu' else 'Validation (not used)', Y['val'], set(PHOTO_KEY['val'])), ('Test', Y['test'], set(PHOTO_KEY['test']))]
DT = [r'\begin{table}[!htbp]', r'\centering', rf'\caption{{Data used ({"ECU-MSS, photo-level validation split" if DATASET == "ecu" else "DeepSeagrass, official splits"}): patches per class and source photographs. '
      rf'The unlabeled pool is the training part minus the $K$ labeled patches per class ({", ".join(f"{v:,}" for v in _pool_n.values())} patches for $K$ = {", ".join(map(str, K_LIST))}).}}', r'\label{tab:data}', r'\footnotesize',
      r'\begin{tabular}{@{}l' + 'r' * (N_CLS + 2) + r'@{}}', r'\toprule', 'Split & ' + ' & '.join(CLASSES) + r' & Total & Photographs \\', r'\midrule'] + \
     [f'{n} & ' + ' & '.join(f'{int((y_ == c).sum()):,}' for c in range(N_CLS)) + f' & {len(y_):,} & {len(ph):,}' + r' \\' for n, y_, ph in DS_rows] + [r'\bottomrule', r'\end{tabular}', r'\end{table}']
tex.append('\n'.join(DT))
open(os.path.join(OUT_DIR, 'latex_tables.tex'), 'w').write('\n\n'.join(tex)); print('\n\n'.join(tex))

# ---- figures -------------------------------------------------------------------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(7, 4.2))
for v, lab in [('baseline', 'Baseline'), ('random_pseudo', 'Random'), ('conf_pseudo', 'Conf'), ('hybrid_pseudo', 'Hybrid (original)'), (NEW, 'Proposed'), (ORACLE_NEW, 'Verified (oracle)')]:
    s = SUM[SUM.variant == v].sort_values('K'); ax.errorbar(s.K, s['mean'], s['sd'], marker='o', capsize=3, label=lab, ls='--' if 'oracle' in v else '-')
ax.set_xscale('log'); ax.set_xticks(K_LIST); ax.set_xticklabels(K_LIST); ax.xaxis.set_minor_formatter(matplotlib.ticker.NullFormatter()); ax.set_xlabel('shots per class'); ax.set_ylabel(f'accuracy (%) on {EVAL_SPLIT}'); ax.grid(alpha=.3); ax.legend(fontsize=8); plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'fig_accuracy_vs_shots.png'), dpi=200); plt.show()
fig, ax = plt.subplots(figsize=(7, 4.2)); dpp = []
for v in ['random_pseudo', 'conf_pseudo', 'dissim_pseudo', 'hybrid_pseudo', NEW]:
    m_ = [mean_diff(ALL, v, 'baseline', K) for K in K_LIST]
    lo_hi = [tci(acc_of(K, v).sub(acc_of(K, 'baseline')).dropna().values) for K in K_LIST]
    ax.errorbar(np.array(K_LIST) * (1 + 0.04 * len(dpp)), m_, [np.array(m_) - [l for l, h in lo_hi], [h for l, h in lo_hi] - np.array(m_)], marker='o', capsize=3, label=v); dpp.append(v)
ax.axhline(0, color='k', lw=0.8); ax.set_xscale('log'); ax.set_xticks(K_LIST); ax.set_xticklabels(K_LIST); ax.xaxis.set_minor_formatter(matplotlib.ticker.NullFormatter()); ax.set_xlabel('shots per class'); ax.set_ylabel('paired difference vs baseline (pp, 95% CI)'); ax.grid(alpha=.3); ax.legend(fontsize=7); plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'fig_diff_vs_baseline.png'), dpi=200); plt.show()
fig, axs = plt.subplots(1, 4, figsize=(15, 3.6), sharey=True)
for a, n in zip(axs, CLASSES):
    for v, lab in [('baseline', 'Baseline'), ('random_pseudo', 'Random'), (NEW, 'Proposed')]:
        g = RES[RES.variant == v].groupby('K')[f'f1_{n}'].agg(['mean', 'std']); a.errorbar(g.index, 100 * g['mean'], 100 * g['std'], marker='o', capsize=3, label=lab)
    a.set_title(n); a.set_xscale('log'); a.set_xticks(K_LIST); a.set_xticklabels(K_LIST); a.xaxis.set_minor_formatter(matplotlib.ticker.NullFormatter()); a.grid(alpha=.3)
axs[0].set_ylabel('F1 (%)'); axs[0].legend(fontsize=7); plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, 'fig_per_class_f1.png'), dpi=200); plt.show()
# ---- v4: mean row-normalised confusion matrices (baseline vs proposed, smallest and largest K) ---------------------------------------------------------
def mean_cm(v, K):
    cms = [np.array(json.loads(c), float) for c in RES[(RES.variant == v) & (RES.K == K)]['cm'].dropna()]
    if not cms: return None
    cms = [c / np.maximum(c.sum(1, keepdims=True), 1) for c in cms]; return 100 * np.mean(cms, 0)
fig, axs = plt.subplots(2, 2, figsize=(9, 8))
for a, (v, K) in zip(axs.ravel(), [('baseline', K_LIST[0]), (NEW, K_LIST[0]), ('baseline', K_LIST[-1]), (NEW, K_LIST[-1])]):
    cm = mean_cm(v, K)
    if cm is None: a.axis('off'); continue
    a.imshow(cm, cmap='Blues', vmin=0, vmax=100); a.set_title(f"{'Baseline' if v == 'baseline' else 'Proposed'}, {K}-shot", fontsize=10)
    for i in range(N_CLS):
        for j in range(N_CLS): a.text(j, i, f'{cm[i, j]:.0f}', ha='center', va='center', color='white' if cm[i, j] > 60 else 'black', fontsize=9)
    a.set_xticks(range(N_CLS)); a.set_xticklabels(CLASSES, rotation=30, fontsize=8); a.set_yticks(range(N_CLS)); a.set_yticklabels(CLASSES, fontsize=8)
    a.set_xlabel('predicted'); a.set_ylabel('true')
plt.suptitle(f'Row-normalised confusion (%), mean over {len(SEEDS)} seeds ({nseed})', fontsize=10); plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, 'fig_confusion.png'), dpi=200); plt.show()

# ---- v4: where do the methods put their added images?  t-SNE of frozen backbone features for one seed (K = second smallest) ------------------------------
try:
    from sklearn.manifold import TSNE
    Kv = K_LIST[1] if len(K_LIST) > 1 else K_LIST[0]; s0 = SEEDS[0]; ctx = get_ctx(Kv, s0); cfgs = variants(Kv); rng_t = np.random.default_rng(0)
    bg = ctx['pool'][rng_t.choice(len(ctx['pool']), size=min(1500, len(ctx['pool'])), replace=False)]
    _ch = {lab: halt_expand(ctx, cfgs[v], s0)[0] for v, lab in [('conf_pseudo', 'Conf'), ('random_pseudo', 'Random'), (NEW, 'Proposed')]}   # chosen pool rows per class
    sel = {lab: [int(ctx['pool'][i]) for c in range(N_CLS) for i in ch[c]] for lab, ch in _ch.items()}
    okd = {lab: np.array([Y['train'][int(ctx['pool'][i])] == c for c in range(N_CLS) for i in ch[c]]) for lab, ch in _ch.items()}
    shots = [int(i) for l in ctx['shots'] for i in l]; allidx = np.unique(np.concatenate([bg, shots] + [np.array(v_) for v_ in sel.values()]))
    Z = TSNE(n_components=2, init='pca', perplexity=30, random_state=0).fit_transform(CLIPN_TR_NP[allidx]); pos = {int(i): Z[j] for j, i in enumerate(allidx)}
    fig, axs = plt.subplots(1, 3, figsize=(15, 4.6)); cols = plt.cm.tab10(np.arange(N_CLS))
    for a, (lab, idx) in zip(axs, sel.items()):
        P_ = np.array([pos[int(i)] for i in bg]); a.scatter(P_[:, 0], P_[:, 1], c=cols[Y['train'][bg]], s=4, alpha=0.15)
        S_ = np.array([pos[i] for i in shots]); a.scatter(S_[:, 0], S_[:, 1], c=cols[Y['train'][shots]], s=140, marker='*', edgecolor='k')
        A_ = np.array([pos[i] for i in idx]); ok_ = okd[lab]
        a.scatter(A_[:, 0], A_[:, 1], facecolors='none', edgecolors=['g' if o else 'r' for o in ok_], s=60, linewidths=1.6)
        a.set_title(f'{lab}: {len(idx)} added images, {100 * ok_.mean():.0f}% correct'); a.set_xticks([]); a.set_yticks([])
    from matplotlib.lines import Line2D
    hands = [Line2D([], [], marker='*', ls='', mfc='w', mec='k', ms=12, label='labeled shots'), Line2D([], [], marker='o', ls='', mfc='none', mec='g', ms=8, mew=1.6, label='added, correct pseudo-label'),
             Line2D([], [], marker='o', ls='', mfc='none', mec='r', ms=8, mew=1.6, label='added, wrong pseudo-label')] + [Line2D([], [], marker='o', ls='', color=cols[c], ms=6, label=CLASSES[c]) for c in range(N_CLS)]
    fig.legend(handles=hands, loc='lower center', ncol=len(hands), fontsize=8, frameon=False); fig.subplots_adjust(bottom=0.14); plt.suptitle(f't-SNE of frozen {BB_LABEL} features: pool sample, labeled shots and the images each method adds ({Kv}-shot, seed {s0}, {nseed})', fontsize=10)
    plt.tight_layout(rect=(0, 0.08, 1, 1)); plt.savefig(os.path.join(OUT_DIR, 'fig_selection_map.png'), dpi=200, bbox_inches='tight'); plt.show()
except Exception as e:
    print('selection map skipped:', type(e).__name__, e)
json.dump({'stage': STAGE, 'eval_split': EVAL_SPLIT, 'seeds': SEEDS, 'tau': TAU, 'w_conf': W_CONF, 'tau_gate': TAU_GATE, 'w_norm': W_NORM, 'new_method': NEW_METHOD, 'val_mode': VAL_MODE, 'val_photo_frac': VAL_PHOTO_FRAC, 'val_split_seed': VAL_SPLIT_SEED, 'val_split': VAL_SPLIT_INFO,
           'hp': HP, 'n_aug': N_AUG, 'dataset': DATASET, 'backbone': BB_LABEL, 'backbone_model': BB_NAME, 'leakage': LEAK, 'hardware': dev_name, 'torch': torch.__version__}, open(os.path.join(OUT_DIR, 'run_config.json'), 'w'), indent=2, default=str)
print('all outputs saved in', OUT_DIR)


## Cross-combination summary (final stage): ECU-MSS / DeepSeagrass x CLIP / DINOv3 in one table and one figure

In [ ]:
# MAIN_SHARD_ONLY
# ===============================================================================
# CROSS-COMBINATION SUMMARY (final stage): one table + one figure with the TEST results of every dataset x backbone combination that has finished.
# Re-run at the end of every final stage, so after the last combination it contains all four. Output: <PROJECT>\summary_v4\
# ===============================================================================
if STAGE == 'final':
    SUMDIR = os.path.join(PROJECT, f'summary_{RUN_TAG}'); os.makedirs(SUMDIR, exist_ok=True)
    LAB = {'ecu': 'ECU-MSS', 'deepseagrass': 'DeepSeagrass', 'clip': 'CLIP ViT-B/32', 'dinov3': 'DINOv3 ViT-B/16'}
    SHOW = [('baseline', 'Baseline'), ('random_pseudo', 'Random'), ('conf_pseudo', 'Conf'), (None, 'Proposed'), ('meanteacher', 'Mean Teacher')]
    rows, tex_rows, found = [], [], {}
    for ds in ('ecu', 'deepseagrass'):
        for bb in ('clip', 'dinov3'):
            d = os.path.join(combo_root(ds, bb), 'final', RUN_TAG); fs = [os.path.join(d, f) for f in ('summary_all_methods.csv', 'paired_tests_acc.csv', 'run_config.json')]
            if not all(os.path.exists(f) for f in fs): print(f'summary: {ds} + {bb} not finished yet ({d})'); continue
            S_, P_ = pd.read_csv(fs[0]), pd.read_csv(fs[1]); new = json.load(open(fs[2]))['new_method']; found[(ds, bb)] = (S_, new)
            for K in sorted(S_.K.unique()):
                def mean_sd(v):
                    r = S_[(S_.K == K) & (S_.variant == v)]; return (float(r['mean'].iloc[0]), float(r['sd'].iloc[0])) if len(r) else (np.nan, np.nan)
                def pdiff(b):
                    r = P_[(P_.K == K) & (P_.A == new) & (P_.B == b) & (P_.family == 'primary')]
                    if not len(r): return np.nan, np.nan, np.nan, ''
                    star = '$^{*}$' if r.p_wilcoxon_holm.iloc[0] < 0.05 else ''
                    return float(r.mean_diff_pp.iloc[0]), float(r.ci_lo.iloc[0]), float(r.ci_hi.iloc[0]), star
                vals = {lab: mean_sd(new if v is None else v) for v, lab in SHOW}; db, dr = pdiff('baseline'), pdiff('random_pseudo')
                rows.append(dict(dataset=LAB[ds], backbone=LAB[bb], K=int(K), proposed=new, **{f'{lab}_mean': m for lab, (m, s) in vals.items()}, **{f'{lab}_sd': s for lab, (m, s) in vals.items()},
                                 diff_vs_baseline_pp=db[0], ci_vs_baseline=f'[{db[1]:.2f}, {db[2]:.2f}]', diff_vs_random_pp=dr[0], ci_vs_random=f'[{dr[1]:.2f}, {dr[2]:.2f}]'))
                tex_rows.append(f"{LAB[ds]} & {LAB[bb]} & {int(K)} & " + ' & '.join(f'{m:.2f} $\\pm$ {s:.2f}' for m, s in vals.values()) + f" & {db[0]:+.2f}{db[3]} & {dr[0]:+.2f}{dr[3]}" + r' \\')
    if rows:
        SUMT = pd.DataFrame(rows); SUMT.to_csv(os.path.join(SUMDIR, 'summary_all_combinations.csv'), index=False)
        T_ = [r'\begin{table*}[!htbp]', r'\centering',
              r'\caption{Test accuracy (\%, mean $\pm$ SD over seeds) for both datasets and both frozen backbones. ECU-MSS: all choices made on the ECU-MSS photo-level validation split; '
              r'DeepSeagrass: every setting frozen from ECU-MSS (no tuning on DeepSeagrass). Last two columns: paired difference of the proposed selection vs.\ the baseline and vs.\ random '
              r'expansion; $^{*}$: Holm-adjusted Wilcoxon $p<0.05$ within the 12 pre-specified primary comparisons of that combination.}', r'\label{tab:all_combinations}', r'\scriptsize',
              r'\setlength{\tabcolsep}{3pt}', r'\begin{tabular}{@{}llc' + 'c' * (len(SHOW) + 2) + r'@{}}', r'\toprule',
              'Dataset & Backbone & $K$ & ' + ' & '.join(lab for _, lab in SHOW) + r' & $\Delta$ Base & $\Delta$ Rand \\', r'\midrule'] + tex_rows + [r'\bottomrule', r'\end{tabular}', r'\end{table*}']
        open(os.path.join(SUMDIR, 'summary_all_combinations.tex'), 'w').write('\n'.join(T_)); print('\n'.join(T_))
        fig, axs = plt.subplots(1, len(found), figsize=(4.2 * len(found), 3.8), squeeze=False)
        for a, ((ds, bb), (S_, new)) in zip(axs[0], found.items()):
            for v, lab in SHOW:
                s = S_[S_.variant == (new if v is None else v)].sort_values('K')
                if len(s): a.errorbar(s.K, s['mean'], s['sd'], marker='o', capsize=3, label=lab, lw=2.2 if v is None else 1.2)
            a.set_xscale('log'); ks = sorted(S_.K.unique()); a.set_xticks(ks); a.set_xticklabels(ks); a.xaxis.set_minor_formatter(matplotlib.ticker.NullFormatter()); a.set_title(f'{LAB[ds]}\n{LAB[bb]}', fontsize=9); a.grid(alpha=.3); a.set_xlabel('shots per class')
        axs[0][0].set_ylabel('test accuracy (%)'); axs[0][0].legend(fontsize=7); plt.tight_layout(); plt.savefig(os.path.join(SUMDIR, 'fig_all_combinations.png'), dpi=200); plt.show()
        print(f'cross-combination summary ({len(found)} of 4 combinations) saved in', SUMDIR)
